In [9]:
import time
from pynq import Overlay
import time
import numpy as np
from pynq import Xlnk
import struct
from scipy.misc import imread
import cv2
import os
xlnk = Xlnk()
overlay = Overlay('./shufflenet.bit')
print("Overlay downloaded successfully!")

Overlay downloaded successfully!


In [10]:
conv_ip = overlay.conv_bn_relu_0
pool_ip = overlay.pooling_0
channel_shuffle_ip = overlay.channel_shuffle_0
fc_ip = overlay.fully_connected_0
    
# 卷积IP核驱动函数    
def hwConv(conv, Conv_type, Ksize, Stride, Padding, feat_in, Conv_w, BN_mean, BN_val, BN_gamma, BN_beta, feat_out):
    conv.write(0x10, Conv_type)                  # 卷积类型0->普通卷积，1->深度卷积
    conv.write(0x30, feat_in.shape[2])           # 输入特征图通道数
    conv.write(0x18, feat_in.shape[0])           # 输入特征图高度
    conv.write(0x38, feat_in.shape[1])           # 输入特征图宽度
    conv.write(0x40, Ksize)                      # 卷积核高度
    conv.write(0x28, Stride)                     # 卷积步长
    conv.write(0x48, Padding)                    # 输入特征图是否需要padding
    
    conv.write(0x20, feat_out.shape[2])          # 输入特征图通道数
    conv.write(0x50, feat_in.physical_address)   # 输入特征图地址
    conv.write(0x5c, Conv_w.physical_address)    # 卷积核地址
    conv.write(0x68, BN_mean.physical_address)   # 归一化参数
    conv.write(0x74, BN_val.physical_address)    # 归一化参数
    conv.write(0x80, BN_gamma.physical_address)  # 归一化参数
    conv.write(0x8c, BN_beta.physical_address)   # 归一化参数
    conv.write(0x98, feat_out.physical_address)  # 输出特征图地址

    conv.write(0, (conv.read(0) & 0x80) | 0x01)
    tp = conv.read(0)
    while not ((tp >> 1) & 0x1):
        tp = conv.read(0)


# 池化IP核驱动函数
def hwPool(pool, Ksize, Stride, Padding, Pool_type, feat_in, feat_out):
    pool.write(0x10, feat_in.shape[2])           # 输入特征图通道数
    pool.write(0x30, feat_in.shape[0])           # 输入特征图高度
    pool.write(0x18, feat_in.shape[1])           # 输入特征图宽度
    pool.write(0x38, Ksize)                      # 池化核高度
    pool.write(0x20, Stride)                     # 卷积步长
    pool.write(0x40, Padding)                    # 卷积填充
    pool.write(0x28, Pool_type)                  # 池化类型

    pool.write(0x48, feat_in.physical_address)   # 输入特征图地址
    pool.write(0x54, feat_out.physical_address)  # 输出特征图地址

    pool.write(0, (pool.read(0) & 0x80) | 0x01)
    while not ((pool.read(0) >> 1) & 0x1):
        pass

# 通道混洗IP核驱动函数
def hwch_shuffle(channel_shuffle, Groups, feat_in, feat_out):
    channel_shuffle.write(0x10, feat_in.shape[2])           # 输入特征图通道数
    channel_shuffle.write(0x20, feat_in.shape[1])           # 输入特征图尺寸

    channel_shuffle.write(0x18, Groups)                     # 分组数==2

    channel_shuffle.write(0x28, feat_in.physical_address)   # 输入特征图地址
    channel_shuffle.write(0x34, feat_out.physical_address)  # 输出特征图地址

    channel_shuffle.write(0, (channel_shuffle.read(0) & 0x80) | 0x01)
    while not ((channel_shuffle.read(0) >> 1) & 0x1):
        pass

# 全连接层IP核驱动函数
def hwfc(fc_connect, feat_in, W, Bias, feat_out):
    fc_connect.write(0x10, feat_in.shape[0])           # 输入特征图通道数
    fc_connect.write(0x18, feat_out.shape[0])          # 输出特征图通道数

    fc_connect.write(0x20, feat_in.physical_address)   # 输入特征图地址
    fc_connect.write(0x2c, W.physical_address)         # 权重参数地址
    fc_connect.write(0x38, Bias.physical_address)      # 偏置参数地址
    fc_connect.write(0x44, feat_out.physical_address)  # 输出特征图地址

    fc_connect.write(0, (fc_connect.read(0) & 0x80) | 0x01)
    while not ((fc_connect.read(0) >> 1) & 0x1):
        pass


def readbinfile(filename, size):
    f = open(filename, "rb")
    z = []
    for j in range(size):
        data = f.read(4)
        data_float = struct.unpack("f", data)[0]
        z.append(data_float)
    f.close()
    z = np.array(z, dtype = np.float32)
    return z
print(" successfully!")

 successfully!


In [11]:
#输入iamge
image = xlnk.cma_array(shape = (128, 128, 3), cacheable = 0, dtype = np.float32)
#Conv1卷积核，BN参数
w_conv1 = xlnk.cma_array(shape = (24, 3, 3, 3), cacheable = 0, dtype = np.float32)
W_conv1 = readbinfile("./data/conv1_0_weight.bin", 24*3*3*3)
W_conv1 = W_conv1.reshape((24, 3, 3, 3))
xlnk.cma_memcopy(w_conv1, W_conv1, 24*3*3*3*4)

bn_mean_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
BN_mean_conv1 = readbinfile("./data/conv1_1_running_mean.bin", 24)
BN_mean_conv1 = BN_mean_conv1.reshape((24))
xlnk.cma_memcopy(bn_mean_conv1, BN_mean_conv1, 24*4)

bn_val_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
BN_val_conv1 = readbinfile("./data/conv1_1_running_var.bin", 24)
BN_val_conv1 = BN_val_conv1.reshape((24))
xlnk.cma_memcopy(bn_val_conv1, BN_val_conv1, 24*4)

bn_gamma_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
BN_gamma_conv1 = readbinfile("./data/conv1_1_weight.bin", 24)
BN_gamma_conv1 = BN_gamma_conv1.reshape((24))
xlnk.cma_memcopy(bn_gamma_conv1, BN_gamma_conv1, 24*4)

bn_beta_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
BN_beta_conv1 = readbinfile("./data/conv1_1_bias.bin", 24)
BN_beta_conv1 = BN_beta_conv1.reshape((24))
xlnk.cma_memcopy(bn_beta_conv1, BN_beta_conv1, 24*4)
#Conv1输出64X64x24
out_conv1 = xlnk.cma_array(shape = (64, 64, 24), cacheable = 0, dtype = np.float32)
#Maxpool输出32x32x24
out_maxpool = xlnk.cma_array(shape = (32, 32, 24), cacheable = 0, dtype = np.float32)

print(" successfully!")

 successfully!


 ## <span style="font-size: 24px;">***stage 2 代码块（核心部分）***</span>

In [12]:
'''                                          stage2                                                  '''
#stage2下采样单元-分支1-3x3深度卷积输出
out_s2s_b1_convdw = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2下采样单元-分支1-3x3深度卷积核，BN参数
s2s_b1_w_convdw = xlnk.cma_array(shape = (24, 3, 3), cacheable = 0, dtype = np.float32)
S2s_b1_w_convdw = readbinfile("./data/stage2_0_branch1_0_weight.bin", 24*3*3)
S2s_b1_w_convdw = S2s_b1_w_convdw.reshape((24, 3, 3))
xlnk.cma_memcopy(s2s_b1_w_convdw, S2s_b1_w_convdw, 24*3*3*4)

s2s_b1_bn_mean_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b1_bn_mean_convdw = readbinfile("./data/stage2_0_branch1_1_running_mean.bin", 24)
S2s_b1_bn_mean_convdw = S2s_b1_bn_mean_convdw.reshape((24))
xlnk.cma_memcopy(s2s_b1_bn_mean_convdw, S2s_b1_bn_mean_convdw, 24*4)

s2s_b1_bn_val_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b1_bn_val_convdw = readbinfile("./data/stage2_0_branch1_1_running_var.bin", 24)
S2s_b1_bn_val_convdw = S2s_b1_bn_val_convdw.reshape((24))
xlnk.cma_memcopy(s2s_b1_bn_val_convdw, S2s_b1_bn_val_convdw, 24*4)

s2s_b1_bn_gamma_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b1_bn_gamma_convdw = readbinfile("./data/stage2_0_branch1_1_weight.bin", 24)
S2s_b1_bn_gamma_convdw = S2s_b1_bn_gamma_convdw.reshape((24))
xlnk.cma_memcopy(s2s_b1_bn_gamma_convdw, S2s_b1_bn_gamma_convdw, 24*4)

s2s_b1_bn_beta_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b1_bn_beta_convdw = readbinfile("./data/stage2_0_branch1_1_bias.bin", 24)
S2s_b1_bn_beta_convdw = S2s_b1_bn_beta_convdw.reshape((24))
xlnk.cma_memcopy(s2s_b1_bn_beta_convdw, S2s_b1_bn_beta_convdw, 24*4)

#stage2下采样单元-分支1-1x1普通卷积输出
out_s2s_b1_conv1 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2下采样单元-分支1-1x1普通卷积核，BN参数
s2s_b1_w_conv1 = xlnk.cma_array(shape = (24, 24, 1, 1), cacheable = 0, dtype = np.float32)
S2s_b1_w_conv1 = readbinfile("./data/stage2_0_branch1_2_weight.bin", 24*24*1*1)
S2s_b1_w_conv1 = S2s_b1_w_conv1.reshape((24, 24, 1, 1))
xlnk.cma_memcopy(s2s_b1_w_conv1, S2s_b1_w_conv1, 24*24*1*1*4)

s2s_b1_bn_mean_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b1_bn_mean_conv1 = readbinfile("./data/stage2_0_branch1_3_running_mean.bin", 24)
S2s_b1_bn_mean_conv1 = S2s_b1_bn_mean_conv1.reshape((24))
xlnk.cma_memcopy(s2s_b1_bn_mean_conv1, S2s_b1_bn_mean_conv1, 24*4)

s2s_b1_bn_val_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b1_bn_val_conv1 = readbinfile("./data/stage2_0_branch1_3_running_var.bin", 24)
S2s_b1_bn_val_conv1 = S2s_b1_bn_val_conv1.reshape((24))
xlnk.cma_memcopy(s2s_b1_bn_val_conv1, S2s_b1_bn_val_conv1, 24*4)

s2s_b1_bn_gamma_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b1_bn_gamma_conv1 = readbinfile("./data/stage2_0_branch1_3_weight.bin", 24)
S2s_b1_bn_gamma_conv1 = S2s_b1_bn_gamma_conv1.reshape((24))
xlnk.cma_memcopy(s2s_b1_bn_gamma_conv1, S2s_b1_bn_gamma_conv1, 24*4)

s2s_b1_bn_beta_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b1_bn_beta_conv1 = readbinfile("./data/stage2_0_branch1_3_bias.bin", 24)
S2s_b1_bn_beta_conv1 = S2s_b1_bn_beta_conv1.reshape((24))
xlnk.cma_memcopy(s2s_b1_bn_beta_conv1, S2s_b1_bn_beta_conv1, 24*4)

#stage2下采样单元-分支2-1x1普通卷积1输出
out_s2s_b2_conv1 = xlnk.cma_array(shape = (32, 32, 24), cacheable = 0, dtype = np.float32)
#stage2下采样单元-分支2-1x1普通卷积1核，BN参数
s2s_b2_w_conv1 = xlnk.cma_array(shape = (24, 24, 1, 1), cacheable = 0, dtype = np.float32)
S2s_b2_w_conv1 = readbinfile("./data/stage2_0_branch2_0_weight.bin", 24*24*1*1)
S2s_b2_w_conv1 = S2s_b2_w_conv1.reshape((24, 24, 1, 1))
xlnk.cma_memcopy(s2s_b2_w_conv1, S2s_b2_w_conv1, 24*24*1*1*4)

s2s_b2_bn_mean_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_mean_conv1 = readbinfile("./data/stage2_0_branch2_1_running_mean.bin", 24)
S2s_b2_bn_mean_conv1 = S2s_b2_bn_mean_conv1.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_mean_conv1, S2s_b2_bn_mean_conv1, 24*4)

s2s_b2_bn_val_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_val_conv1 = readbinfile("./data/stage2_0_branch2_1_running_var.bin", 24)
S2s_b2_bn_val_conv1 = S2s_b2_bn_val_conv1.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_val_conv1, S2s_b2_bn_val_conv1, 24*4)

s2s_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_gamma_conv1 = readbinfile("./data/stage2_0_branch2_1_weight.bin", 24)
S2s_b2_bn_gamma_conv1 = S2s_b2_bn_gamma_conv1.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_gamma_conv1, S2s_b2_bn_gamma_conv1, 24*4)

s2s_b2_bn_beta_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_beta_conv1 = readbinfile("./data/stage2_0_branch2_1_bias.bin", 24)
S2s_b2_bn_beta_conv1 = S2s_b2_bn_beta_conv1.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_beta_conv1, S2s_b2_bn_beta_conv1, 24*4)

#stage2下采样单元-分支2-3x3深度卷积输出
out_s2s_b2_convdw = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2下采样单元-分支2-3x3深度卷积核，BN参数
s2s_b2_w_convdw = xlnk.cma_array(shape = (24, 3, 3), cacheable = 0, dtype = np.float32)
S2s_b2_w_convdw = readbinfile("./data/stage2_0_branch2_3_weight.bin", 24*3*3)
S2s_b2_w_convdw = S2s_b2_w_convdw.reshape((24, 3, 3))
xlnk.cma_memcopy(s2s_b2_w_convdw, S2s_b2_w_convdw, 24*3*3*4)

s2s_b2_bn_mean_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_mean_convdw = readbinfile("./data/stage2_0_branch2_4_running_mean.bin", 24)
S2s_b2_bn_mean_convdw = S2s_b2_bn_mean_convdw.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_mean_convdw, S2s_b2_bn_mean_convdw, 24*4)

s2s_b2_bn_val_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_val_convdw = readbinfile("./data/stage2_0_branch2_4_running_var.bin", 24)
S2s_b2_bn_val_convdw = S2s_b2_bn_val_convdw.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_val_convdw, S2s_b2_bn_val_convdw, 24*4)

s2s_b2_bn_gamma_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_gamma_convdw = readbinfile("./data/stage2_0_branch2_4_weight.bin", 24)
S2s_b2_bn_gamma_convdw = S2s_b2_bn_gamma_convdw.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_gamma_convdw, S2s_b2_bn_gamma_convdw, 24*4)

s2s_b2_bn_beta_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_beta_convdw = readbinfile("./data/stage2_0_branch2_4_bias.bin", 24)
S2s_b2_bn_beta_convdw = S2s_b2_bn_beta_convdw.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_beta_convdw, S2s_b2_bn_beta_convdw, 24*4)

#stage2下采样单元-分支2-1x1普通卷积输出2
out_s2s_b2_conv2 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2下采样单元-分支2-1x1普通卷积核2，BN参数
s2s_b2_w_conv2 = xlnk.cma_array(shape = (24, 24, 1, 1), cacheable = 0, dtype = np.float32)
S2s_b2_w_conv2 = readbinfile("./data/stage2_0_branch2_5_weight.bin", 24*24*1*1)
S2s_b2_w_conv2 = S2s_b2_w_conv2.reshape((24, 24, 1, 1))
xlnk.cma_memcopy(s2s_b2_w_conv2, S2s_b2_w_conv2, 24*24*1*1*4)

s2s_b2_bn_mean_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_mean_conv2 = readbinfile("./data/stage2_0_branch2_6_running_mean.bin", 24)
S2s_b2_bn_mean_conv2 = S2s_b2_bn_mean_conv2.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_mean_conv2, S2s_b2_bn_mean_conv2, 24*4)

s2s_b2_bn_val_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_val_conv2 = readbinfile("./data/stage2_0_branch2_6_running_var.bin", 24)
S2s_b2_bn_val_conv2 = S2s_b2_bn_val_conv2.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_val_conv2, S2s_b2_bn_val_conv2, 24*4)

s2s_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_gamma_conv2 = readbinfile("./data/stage2_0_branch2_6_weight.bin", 24)
S2s_b2_bn_gamma_conv2 = S2s_b2_bn_gamma_conv2.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_gamma_conv2, S2s_b2_bn_gamma_conv2, 24*4)

s2s_b2_bn_beta_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2s_b2_bn_beta_conv2 = readbinfile("./data/stage2_0_branch2_6_bias.bin", 24)
S2s_b2_bn_beta_conv2 = S2s_b2_bn_beta_conv2.reshape((24))
xlnk.cma_memcopy(s2s_b2_bn_beta_conv2, S2s_b2_bn_beta_conv2, 24*4)

#stage2下采样单元-通道合并，通道混洗
in_s2s_shuff = xlnk.cma_array(shape = (16, 16, 48), cacheable = 0, dtype = np.float32)
out_s2s_shuff = xlnk.cma_array(shape = (16, 16, 48), cacheable = 0, dtype = np.float32)


#基本单元1
#stage2基本单元1-通道拆分
out_s2c1_ch_spilt1 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
out_s2c1_ch_spilt2 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)

#stage2基本单元1-分支1

#stage2基本单元1-分支2-1x1普通卷积1输出
out_s2c1_b2_conv1 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2基本单元1-分支2-1x1普通卷积1核，BN参数
s2c1_b2_w_conv1 = xlnk.cma_array(shape = (24, 24, 1, 1), cacheable = 0, dtype = np.float32)
S2c1_b2_w_conv1 = readbinfile("./data/stage2_1_branch2_0_weight.bin", 24*24*1*1)
S2c1_b2_w_conv1 = S2c1_b2_w_conv1.reshape((24, 24, 1, 1))
xlnk.cma_memcopy(s2c1_b2_w_conv1, S2c1_b2_w_conv1, 24*24*1*1*4)

s2c1_b2_bn_mean_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_mean_conv1 = readbinfile("./data/stage2_1_branch2_1_running_mean.bin", 24)
S2c1_b2_bn_mean_conv1 = S2c1_b2_bn_mean_conv1.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_mean_conv1, S2c1_b2_bn_mean_conv1, 24*4)

s2c1_b2_bn_val_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_val_conv1 = readbinfile("./data/stage2_1_branch2_1_running_var.bin", 24)
S2c1_b2_bn_val_conv1 = S2c1_b2_bn_val_conv1.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_val_conv1, S2c1_b2_bn_val_conv1, 24*4)

s2c1_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_gamma_conv1 = readbinfile("./data/stage2_1_branch2_1_weight.bin", 24)
S2c1_b2_bn_gamma_conv1 = S2c1_b2_bn_gamma_conv1.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_gamma_conv1, S2c1_b2_bn_gamma_conv1, 24*4)

s2c1_b2_bn_beta_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_beta_conv1 = readbinfile("./data/stage2_1_branch2_1_bias.bin", 24)
S2c1_b2_bn_beta_conv1 = S2c1_b2_bn_beta_conv1.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_beta_conv1, S2c1_b2_bn_beta_conv1, 24*4)

#stage2基本单元1-分支2-3x3深度卷积输出
out_s2c1_b2_convdw = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2基本单元1-分支2-3x3深度卷积核，BN参数
s2c1_b2_w_convdw = xlnk.cma_array(shape = (24, 3, 3), cacheable = 0, dtype = np.float32)
S2c1_b2_w_convdw = readbinfile("./data/stage2_1_branch2_3_weight.bin", 24*3*3)
S2c1_b2_w_convdw = S2c1_b2_w_convdw.reshape((24, 3, 3))
xlnk.cma_memcopy(s2c1_b2_w_convdw, S2c1_b2_w_convdw, 24*3*3*4)

s2c1_b2_bn_mean_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_mean_convdw = readbinfile("./data/stage2_1_branch2_4_running_mean.bin", 24)
S2c1_b2_bn_mean_convdw = S2c1_b2_bn_mean_convdw.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_mean_convdw, S2c1_b2_bn_mean_convdw, 24*4)

s2c1_b2_bn_val_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_val_convdw = readbinfile("./data/stage2_1_branch2_4_running_var.bin", 24)
S2c1_b2_bn_val_convdw = S2c1_b2_bn_val_convdw.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_val_convdw, S2c1_b2_bn_val_convdw, 24*4)

s2c1_b2_bn_gamma_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_gamma_convdw = readbinfile("./data/stage2_1_branch2_4_weight.bin", 24)
S2c1_b2_bn_gamma_convdw = S2c1_b2_bn_gamma_convdw.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_gamma_convdw, S2c1_b2_bn_gamma_convdw, 24*4)

s2c1_b2_bn_beta_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_beta_convdw = readbinfile("./data/stage2_1_branch2_4_bias.bin", 24)
S2c1_b2_bn_beta_convdw = S2c1_b2_bn_beta_convdw.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_beta_convdw, S2c1_b2_bn_beta_convdw, 24*4)

#stage2基本单元1-分支2-1x1普通卷积输出2
out_s2c1_b2_conv2 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2基本单元1-分支2-1x1普通卷积核2，BN参数
s2c1_b2_w_conv2 = xlnk.cma_array(shape = (24, 24, 1, 1), cacheable = 0, dtype = np.float32)
S2c1_b2_w_conv2 = readbinfile("./data/stage2_1_branch2_5_weight.bin", 24*24*1*1)
S2c1_b2_w_conv2 = S2c1_b2_w_conv2.reshape((24, 24, 1, 1))
xlnk.cma_memcopy(s2c1_b2_w_conv2, S2c1_b2_w_conv2, 24*24*1*1*4)

s2c1_b2_bn_mean_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_mean_conv2 = readbinfile("./data/stage2_1_branch2_6_running_mean.bin", 24)
S2c1_b2_bn_mean_conv2 = S2c1_b2_bn_mean_conv2.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_mean_conv2, S2c1_b2_bn_mean_conv2, 24*4)

s2c1_b2_bn_val_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_val_conv2 = readbinfile("./data/stage2_1_branch2_6_running_var.bin", 24)
S2c1_b2_bn_val_conv2 = S2c1_b2_bn_val_conv2.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_val_conv2, S2c1_b2_bn_val_conv2, 24*4)

s2c1_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_gamma_conv2 = readbinfile("./data/stage2_1_branch2_6_weight.bin", 24)
S2c1_b2_bn_gamma_conv2 = S2c1_b2_bn_gamma_conv2.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_gamma_conv2, S2c1_b2_bn_gamma_conv2, 24*4)

s2c1_b2_bn_beta_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c1_b2_bn_beta_conv2 = readbinfile("./data/stage2_1_branch2_6_bias.bin", 24)
S2c1_b2_bn_beta_conv2 = S2c1_b2_bn_beta_conv2.reshape((24))
xlnk.cma_memcopy(s2c1_b2_bn_beta_conv2, S2c1_b2_bn_beta_conv2, 24*4)

#stage2基本单元1-通道合并，通道混洗
in_s2c1_shuff = xlnk.cma_array(shape = (16, 16, 48), cacheable = 0, dtype = np.float32)
out_s2c1_shuff = xlnk.cma_array(shape = (16, 16, 48), cacheable = 0, dtype = np.float32)


#基本单元2
#stage2基本单元2-通道拆分
out_s2c2_ch_spilt1 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
out_s2c2_ch_spilt2 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)  

#stage2基本单元2-分支1

#stage2基本单元2-分支2-1x1普通卷积1输出
out_s2c2_b2_conv1 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2基本单元2-分支2-1x1普通卷积1核，BN参数
s2c2_b2_w_conv1 = xlnk.cma_array(shape = (24, 24, 1, 1), cacheable = 0, dtype = np.float32)
S2c2_b2_w_conv1 = readbinfile("./data/stage2_2_branch2_0_weight.bin", 24*24*1*1)
S2c2_b2_w_conv1 = S2c2_b2_w_conv1.reshape((24, 24, 1, 1))
xlnk.cma_memcopy(s2c2_b2_w_conv1, S2c2_b2_w_conv1, 24*24*1*1*4)

s2c2_b2_bn_mean_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_mean_conv1 = readbinfile("./data/stage2_2_branch2_1_running_mean.bin", 24)
S2c2_b2_bn_mean_conv1 = S2c2_b2_bn_mean_conv1.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_mean_conv1, S2c2_b2_bn_mean_conv1, 24*4)

s2c2_b2_bn_val_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_val_conv1 = readbinfile("./data/stage2_2_branch2_1_running_var.bin", 24)
S2c2_b2_bn_val_conv1 = S2c2_b2_bn_val_conv1.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_val_conv1, S2c2_b2_bn_val_conv1, 24*4)

s2c2_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_gamma_conv1 = readbinfile("./data/stage2_2_branch2_1_weight.bin", 24)
S2c2_b2_bn_gamma_conv1 = S2c2_b2_bn_gamma_conv1.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_gamma_conv1, S2c2_b2_bn_gamma_conv1, 24*4)

s2c2_b2_bn_beta_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_beta_conv1 = readbinfile("./data/stage2_2_branch2_1_bias.bin", 24)
S2c2_b2_bn_beta_conv1 = S2c2_b2_bn_beta_conv1.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_beta_conv1, S2c2_b2_bn_beta_conv1, 24*4)

#stage2基本单元2-分支2-3x3深度卷积输出
out_s2c2_b2_convdw = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2基本单元2-分支2-3x3深度卷积核，BN参数
s2c2_b2_w_convdw = xlnk.cma_array(shape = (24, 3, 3), cacheable = 0, dtype = np.float32)
S2c2_b2_w_convdw = readbinfile("./data/stage2_2_branch2_3_weight.bin", 24*3*3)
S2c2_b2_w_convdw = S2c2_b2_w_convdw.reshape((24, 3, 3))
xlnk.cma_memcopy(s2c2_b2_w_convdw, S2c2_b2_w_convdw, 24*3*3*4)

s2c2_b2_bn_mean_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_mean_convdw = readbinfile("./data/stage2_2_branch2_4_running_mean.bin", 24)
S2c2_b2_bn_mean_convdw = S2c2_b2_bn_mean_convdw.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_mean_convdw, S2c2_b2_bn_mean_convdw, 24*4)

s2c2_b2_bn_val_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_val_convdw = readbinfile("./data/stage2_2_branch2_4_running_var.bin", 24)
S2c2_b2_bn_val_convdw = S2c2_b2_bn_val_convdw.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_val_convdw, S2c2_b2_bn_val_convdw, 24*4)

s2c2_b2_bn_gamma_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_gamma_convdw = readbinfile("./data/stage2_2_branch2_4_weight.bin", 24)
S2c2_b2_bn_gamma_convdw = S2c2_b2_bn_gamma_convdw.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_gamma_convdw, S2c2_b2_bn_gamma_convdw, 24*4)

s2c2_b2_bn_beta_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_beta_convdw = readbinfile("./data/stage2_2_branch2_4_bias.bin", 24)
S2c2_b2_bn_beta_convdw = S2c2_b2_bn_beta_convdw.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_beta_convdw, S2c2_b2_bn_beta_convdw, 24*4)

#stage2基本单元2-分支2-1x1普通卷积输出2
out_s2c2_b2_conv2 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2基本单元2-分支2-1x1普通卷积核2，BN参数
s2c2_b2_w_conv2 = xlnk.cma_array(shape = (24, 24, 1, 1), cacheable = 0, dtype = np.float32)
S2c2_b2_w_conv2 = readbinfile("./data/stage2_2_branch2_5_weight.bin", 24*24*1*1)
S2c2_b2_w_conv2 = S2c2_b2_w_conv2.reshape((24, 24, 1, 1))
xlnk.cma_memcopy(s2c2_b2_w_conv2, S2c2_b2_w_conv2, 24*24*1*1*4)

s2c2_b2_bn_mean_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_mean_conv2 = readbinfile("./data/stage2_2_branch2_6_running_mean.bin", 24)
S2c2_b2_bn_mean_conv2 = S2c2_b2_bn_mean_conv2.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_mean_conv2, S2c2_b2_bn_mean_conv2, 24*4)

s2c2_b2_bn_val_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_val_conv2 = readbinfile("./data/stage2_2_branch2_6_running_var.bin", 24)
S2c2_b2_bn_val_conv2 = S2c2_b2_bn_val_conv2.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_val_conv2, S2c2_b2_bn_val_conv2, 24*4)

s2c2_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_gamma_conv2 = readbinfile("./data/stage2_2_branch2_6_weight.bin", 24)
S2c2_b2_bn_gamma_conv2 = S2c2_b2_bn_gamma_conv2.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_gamma_conv2, S2c2_b2_bn_gamma_conv2, 24*4)

s2c2_b2_bn_beta_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c2_b2_bn_beta_conv2 = readbinfile("./data/stage2_2_branch2_6_bias.bin", 24)
S2c2_b2_bn_beta_conv2 = S2c2_b2_bn_beta_conv2.reshape((24))
xlnk.cma_memcopy(s2c2_b2_bn_beta_conv2, S2c2_b2_bn_beta_conv2, 24*4)

#stage2基本单元2-通道合并，通道混洗
in_s2c2_shuff = xlnk.cma_array(shape = (16, 16, 48), cacheable = 0, dtype = np.float32)
out_s2c2_shuff = xlnk.cma_array(shape = (16, 16, 48), cacheable = 0, dtype = np.float32)


#基本单元3
#stage2基本单元3-通道拆分
out_s2c3_ch_spilt1 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
out_s2c3_ch_spilt2 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)

#stage2基本单元3-分支1

#stage2基本单元3-分支2-1x1普通卷积1输出
out_s2c3_b2_conv1 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2基本单元3-分支2-1x1普通卷积1核，BN参数
s2c3_b2_w_conv1 = xlnk.cma_array(shape = (24, 24, 1, 1), cacheable = 0, dtype = np.float32)
S2c3_b2_w_conv1 = readbinfile("./data/stage2_3_branch2_0_weight.bin", 24*24*1*1)
S2c3_b2_w_conv1 = S2c3_b2_w_conv1.reshape((24, 24, 1, 1))
xlnk.cma_memcopy(s2c3_b2_w_conv1, S2c3_b2_w_conv1, 24*24*1*1*4)

s2c3_b2_bn_mean_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_mean_conv1 = readbinfile("./data/stage2_3_branch2_1_running_mean.bin", 24)
S2c3_b2_bn_mean_conv1 = S2c3_b2_bn_mean_conv1.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_mean_conv1, S2c3_b2_bn_mean_conv1, 24*4)

s2c3_b2_bn_val_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_val_conv1 = readbinfile("./data/stage2_3_branch2_1_running_var.bin", 24)
S2c3_b2_bn_val_conv1 = S2c3_b2_bn_val_conv1.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_val_conv1, S2c3_b2_bn_val_conv1, 24*4)

s2c3_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_gamma_conv1 = readbinfile("./data/stage2_3_branch2_1_weight.bin", 24)
S2c3_b2_bn_gamma_conv1 = S2c3_b2_bn_gamma_conv1.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_gamma_conv1, S2c3_b2_bn_gamma_conv1, 24*4)

s2c3_b2_bn_beta_conv1 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_beta_conv1 = readbinfile("./data/stage2_3_branch2_1_bias.bin", 24)
S2c3_b2_bn_beta_conv1 = S2c3_b2_bn_beta_conv1.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_beta_conv1, S2c3_b2_bn_beta_conv1, 24*4)

#stage2基本单元3-分支2-3x3深度卷积输出
out_s2c3_b2_convdw = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2基本单元3-分支2-3x3深度卷积核，BN参数
s2c3_b2_w_convdw = xlnk.cma_array(shape = (24, 3, 3), cacheable = 0, dtype = np.float32)
S2c3_b2_w_convdw = readbinfile("./data/stage2_3_branch2_3_weight.bin", 24*3*3)
S2c3_b2_w_convdw = S2c3_b2_w_convdw.reshape((24, 3, 3))
xlnk.cma_memcopy(s2c3_b2_w_convdw, S2c3_b2_w_convdw, 24*3*3*4)

s2c3_b2_bn_mean_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_mean_convdw = readbinfile("./data/stage2_3_branch2_4_running_mean.bin", 24)
S2c3_b2_bn_mean_convdw = S2c3_b2_bn_mean_convdw.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_mean_convdw, S2c3_b2_bn_mean_convdw, 24*4)

s2c3_b2_bn_val_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_val_convdw = readbinfile("./data/stage2_3_branch2_4_running_var.bin", 24)
S2c3_b2_bn_val_convdw = S2c3_b2_bn_val_convdw.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_val_convdw, S2c3_b2_bn_val_convdw, 24*4)

s2c3_b2_bn_gamma_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_gamma_convdw = readbinfile("./data/stage2_3_branch2_4_weight.bin", 24)
S2c3_b2_bn_gamma_convdw = S2c3_b2_bn_gamma_convdw.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_gamma_convdw, S2c3_b2_bn_gamma_convdw, 24*4)

s2c3_b2_bn_beta_convdw = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_beta_convdw = readbinfile("./data/stage2_3_branch2_4_bias.bin", 24)
S2c3_b2_bn_beta_convdw = S2c3_b2_bn_beta_convdw.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_beta_convdw, S2c3_b2_bn_beta_convdw, 24*4)

#stage2基本单元3-分支2-1x1普通卷积输出2
out_s2c3_b2_conv2 = xlnk.cma_array(shape = (16, 16, 24), cacheable = 0, dtype = np.float32)
#stage2基本单元3-分支2-1x1普通卷积核2，BN参数
s2c3_b2_w_conv2 = xlnk.cma_array(shape = (24, 24, 1, 1), cacheable = 0, dtype = np.float32)
S2c3_b2_w_conv2 = readbinfile("./data/stage2_3_branch2_5_weight.bin", 24*24*1*1)
S2c3_b2_w_conv2 = S2c3_b2_w_conv2.reshape((24, 24, 1, 1))
xlnk.cma_memcopy(s2c3_b2_w_conv2, S2c3_b2_w_conv2, 24*24*1*1*4)

s2c3_b2_bn_mean_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_mean_conv2 = readbinfile("./data/stage2_3_branch2_6_running_mean.bin", 24)
S2c3_b2_bn_mean_conv2 = S2c3_b2_bn_mean_conv2.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_mean_conv2, S2c3_b2_bn_mean_conv2, 24*4)

s2c3_b2_bn_val_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_val_conv2 = readbinfile("./data/stage2_3_branch2_6_running_var.bin", 24)
S2c3_b2_bn_val_conv2 = S2c3_b2_bn_val_conv2.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_val_conv2, S2c3_b2_bn_val_conv2, 24*4)

s2c3_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_gamma_conv2 = readbinfile("./data/stage2_3_branch2_6_weight.bin", 24)
S2c3_b2_bn_gamma_conv2 = S2c3_b2_bn_gamma_conv2.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_gamma_conv2, S2c3_b2_bn_gamma_conv2, 24*4)

s2c3_b2_bn_beta_conv2 = xlnk.cma_array(shape = (24), cacheable = 0, dtype = np.float32)
S2c3_b2_bn_beta_conv2 = readbinfile("./data/stage2_3_branch2_6_bias.bin", 24)
S2c3_b2_bn_beta_conv2 = S2c3_b2_bn_beta_conv2.reshape((24))
xlnk.cma_memcopy(s2c3_b2_bn_beta_conv2, S2c3_b2_bn_beta_conv2, 24*4)

#stage2基本单元3-通道合并，通道混洗
in_s2c3_shuff = xlnk.cma_array(shape = (16, 16, 48), cacheable = 0, dtype = np.float32)
out_s2c3_shuff = xlnk.cma_array(shape = (16, 16, 48), cacheable = 0, dtype = np.float32)


print("successfully!")

successfully!


 ## <span style="font-size: 24px;">***stage 3 代码块（核心部分）***</span>

In [13]:
'''                                          stage3                                                  '''
#stage3下采样单元-分支1-3x3深度卷积输出
out_s3s_b1_convdw = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3下采样单元-分支1-3x3深度卷积核，BN参数
s3s_b1_w_convdw = xlnk.cma_array(shape = (48, 3, 3), cacheable = 0, dtype = np.float32)
s3s_b1_bn_mean_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b1_bn_val_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b1_bn_gamma_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b1_bn_beta_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3下采样单元-分支1-1x1普通卷积输出
out_s3s_b1_conv1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3下采样单元-分支1-1x1普通卷积核，BN参数
s3s_b1_w_conv1 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3s_b1_bn_mean_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b1_bn_val_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b1_bn_gamma_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b1_bn_beta_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)

#stage3下采样单元-分支2-1x1普通卷积1输出
out_s3s_b2_conv1 = xlnk.cma_array(shape = (16, 16, 48), cacheable = 0, dtype = np.float32)
#stage3下采样单元-分支2-1x1普通卷积1核，BN参数
s3s_b2_w_conv1 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3s_b2_bn_mean_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b2_bn_val_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b2_bn_beta_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3下采样单元-分支2-3x3深度卷积输出
out_s3s_b2_convdw = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3下采样单元-分支2-3x3深度卷积核，BN参数
s3s_b2_w_convdw = xlnk.cma_array(shape = (48, 3, 3), cacheable = 0, dtype = np.float32)
s3s_b2_bn_mean_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b2_bn_val_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b2_bn_gamma_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b2_bn_beta_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3下采样单元-分支2-1x1普通卷积输出2
out_s3s_b2_conv2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3下采样单元-分支2-1x1普通卷积核2，BN参数
s3s_b2_w_conv2 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3s_b2_bn_mean_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b2_bn_val_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3s_b2_bn_beta_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)

#stage3下采样单元-通道合并，通道混洗
in_s3s_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)
out_s3s_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)


#基本单元1
#stage3基本单元1-通道拆分
out_s3c1_ch_spilt1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
out_s3c1_ch_spilt2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)

#stage3基本单元1-分支1

#stage3基本单元1-分支2-1x1普通卷积1输出
out_s3c1_b2_conv1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元1-分支2-1x1普通卷积1核，BN参数
s3c1_b2_w_conv1 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_mean_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_val_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_beta_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元1-分支2-3x3深度卷积输出
out_s3c1_b2_convdw = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元1-分支2-3x3深度卷积核，BN参数
s3c1_b2_w_convdw = xlnk.cma_array(shape = (48, 3, 3), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_mean_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_val_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_gamma_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_beta_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元1-分支2-1x1普通卷积输出2
out_s3c1_b2_conv2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元1-分支2-1x1普通卷积核2，BN参数
s3c1_b2_w_conv2 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_mean_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_val_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c1_b2_bn_beta_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)

#stage3基本单元1-通道合并，通道混洗
in_s3c1_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)
out_s3c1_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)


#基本单元2
#stage3基本单元2-通道拆分
out_s3c2_ch_spilt1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
out_s3c2_ch_spilt2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)

#stage3基本单元2-分支1

#stage3基本单元2-分支2-1x1普通卷积1输出
out_s3c2_b2_conv1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元2-分支2-1x1普通卷积1核，BN参数
s3c2_b2_w_conv1 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_mean_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_val_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_beta_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元2-分支2-3x3深度卷积输出
out_s3c2_b2_convdw = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元2-分支2-3x3深度卷积核，BN参数
s3c2_b2_w_convdw = xlnk.cma_array(shape = (48, 3, 3), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_mean_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_val_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_gamma_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_beta_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元2-分支2-1x1普通卷积输出2
out_s3c2_b2_conv2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元2-分支2-1x1普通卷积核2，BN参数
s3c2_b2_w_conv2 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_mean_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_val_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c2_b2_bn_beta_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)

#stage3基本单元2-通道合并，通道混洗
in_s3c2_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)
out_s3c2_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)


#基本单元3
#stage3基本单元3-通道拆分
out_s3c3_ch_spilt1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
out_s3c3_ch_spilt2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)

#stage3基本单元3-分支1

#stage3基本单元3-分支2-1x1普通卷积1输出
out_s3c3_b2_conv1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元3-分支2-1x1普通卷积1核，BN参数
s3c3_b2_w_conv1 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_mean_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_val_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_beta_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元3-分支2-3x3深度卷积输出
out_s3c3_b2_convdw = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元3-分支2-3x3深度卷积核，BN参数
s3c3_b2_w_convdw = xlnk.cma_array(shape = (48, 3, 3), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_mean_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_val_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_gamma_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_beta_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元3-分支2-1x1普通卷积输出2
out_s3c3_b2_conv2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元3-分支2-1x1普通卷积核2，BN参数
s3c3_b2_w_conv2 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_mean_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_val_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c3_b2_bn_beta_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)

#stage3基本单元3-通道合并，通道混洗
in_s3c3_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)
out_s3c3_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)


#基本单元4
#stage3基本单元4-通道拆分
out_s3c4_ch_spilt1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
out_s3c4_ch_spilt2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)  

#stage3基本单元4-分支1

#stage3基本单元4-分支2-1x1普通卷积1输出
out_s3c4_b2_conv1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元4-分支2-1x1普通卷积1核，BN参数
s3c4_b2_w_conv1 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_mean_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_val_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_beta_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元4-分支2-3x3深度卷积输出
out_s3c4_b2_convdw = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元4-分支2-3x3深度卷积核，BN参数
s3c4_b2_w_convdw = xlnk.cma_array(shape = (48, 3, 3), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_mean_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_val_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_gamma_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_beta_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元4-分支2-1x1普通卷积输出2
out_s3c4_b2_conv2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元4-分支2-1x1普通卷积核2，BN参数
s3c4_b2_w_conv2 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_mean_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_val_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c4_b2_bn_beta_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)

#stage3基本单元4-通道合并，通道混洗
in_s3c4_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)
out_s3c4_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)


#基本单元5
#stage3基本单元5-通道拆分
out_s3c5_ch_spilt1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
out_s3c5_ch_spilt2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32) 

#stage3基本单元5-分支1

#stage3基本单元5-分支2-1x1普通卷积1输出
out_s3c5_b2_conv1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元5-分支2-1x1普通卷积1核，BN参数
s3c5_b2_w_conv1 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_mean_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_val_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_beta_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元5-分支2-3x3深度卷积输出
out_s3c5_b2_convdw = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元5-分支2-3x3深度卷积核，BN参数
s3c5_b2_w_convdw = xlnk.cma_array(shape = (48, 3, 3), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_mean_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_val_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_gamma_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_beta_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元5-分支2-1x1普通卷积输出2
out_s3c5_b2_conv2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元5-分支2-1x1普通卷积核2，BN参数
s3c5_b2_w_conv2 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_mean_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_val_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c5_b2_bn_beta_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)

#stage3基本单元5-通道合并，通道混洗
in_s3c5_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)
out_s3c5_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)


#基本单元6
#stage3基本单元6-通道拆分
out_s3c6_ch_spilt1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
out_s3c6_ch_spilt2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)

#stage3基本单元6-分支1

#stage3基本单元6-分支2-1x1普通卷积1输出
out_s3c6_b2_conv1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元6-分支2-1x1普通卷积1核，BN参数
s3c6_b2_w_conv1 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_mean_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_val_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_beta_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元6-分支2-3x3深度卷积输出
out_s3c6_b2_convdw = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元6-分支2-3x3深度卷积核，BN参数
s3c6_b2_w_convdw = xlnk.cma_array(shape = (48, 3, 3), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_mean_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_val_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_gamma_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_beta_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元6-分支2-1x1普通卷积输出2
out_s3c6_b2_conv2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元6-分支2-1x1普通卷积核2，BN参数
s3c6_b2_w_conv2 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_mean_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_val_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c6_b2_bn_beta_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)

#stage3基本单元6-通道合并，通道混洗
in_s3c6_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)
out_s3c6_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)


#基本单元7
#stage3基本单元7-通道拆分
out_s3c7_ch_spilt1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
out_s3c7_ch_spilt2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)

#stage3基本单元7-分支1

#stage3基本单元7-分支2-1x1普通卷积1输出
out_s3c7_b2_conv1 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元7-分支2-1x1普通卷积1核，BN参数
s3c7_b2_w_conv1 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_mean_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_val_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_beta_conv1 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元7-分支2-3x3深度卷积输出
out_s3c7_b2_convdw = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元7-分支2-3x3深度卷积核，BN参数
s3c7_b2_w_convdw = xlnk.cma_array(shape = (48, 3, 3), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_mean_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_val_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_gamma_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_beta_convdw = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
#stage3基本单元7-分支2-1x1普通卷积输出2
out_s3c7_b2_conv2 = xlnk.cma_array(shape = (8, 8, 48), cacheable = 0, dtype = np.float32)
#stage3基本单元7-分支2-1x1普通卷积核2，BN参数
s3c7_b2_w_conv2 = xlnk.cma_array(shape = (48, 48, 1, 1), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_mean_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_val_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)
s3c7_b2_bn_beta_conv2 = xlnk.cma_array(shape = (48), cacheable = 0, dtype = np.float32)

#stage3基本单元7-通道合并，通道混洗
in_s3c7_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)
out_s3c7_shuff = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)


S3s_b1_w_convdw = readbinfile("./data/stage3_0_branch1_0_weight.bin", 48*3*3)
S3s_b1_w_convdw = S3s_b1_w_convdw.reshape((48, 3, 3))
xlnk.cma_memcopy(s3s_b1_w_convdw, S3s_b1_w_convdw, 48*3*3*4)
S3s_b1_bn_mean_convdw = readbinfile("./data/stage3_0_branch1_1_running_mean.bin", 48)
xlnk.cma_memcopy(s3s_b1_bn_mean_convdw, S3s_b1_bn_mean_convdw, 48*4)
S3s_b1_bn_val_convdw = readbinfile("./data/stage3_0_branch1_1_running_var.bin", 48)
xlnk.cma_memcopy(s3s_b1_bn_val_convdw, S3s_b1_bn_val_convdw, 48*4)
S3s_b1_bn_gamma_convdw = readbinfile("./data/stage3_0_branch1_1_weight.bin", 48)
xlnk.cma_memcopy(s3s_b1_bn_gamma_convdw, S3s_b1_bn_gamma_convdw, 48*4)
S3s_b1_bn_beta_convdw = readbinfile("./data/stage3_0_branch1_1_bias.bin", 48)
xlnk.cma_memcopy(s3s_b1_bn_beta_convdw, S3s_b1_bn_beta_convdw, 48*4)

S3s_b1_w_conv1 = readbinfile("./data/stage3_0_branch1_2_weight.bin", 48*48*1*1)
S3s_b1_w_conv1 = S3s_b1_w_conv1.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3s_b1_w_conv1, S3s_b1_w_conv1, 48*48*1*1*4)
S3s_b1_bn_mean_conv1 = readbinfile("./data/stage3_0_branch1_3_running_mean.bin", 48)
xlnk.cma_memcopy(s3s_b1_bn_mean_conv1, S3s_b1_bn_mean_conv1, 48*4)
S3s_b1_bn_val_conv1 = readbinfile("./data/stage3_0_branch1_3_running_var.bin", 48)
xlnk.cma_memcopy(s3s_b1_bn_val_conv1, S3s_b1_bn_val_conv1, 48*4)
S3s_b1_bn_gamma_conv1 = readbinfile("./data/stage3_0_branch1_3_weight.bin", 48)
xlnk.cma_memcopy(s3s_b1_bn_gamma_conv1, S3s_b1_bn_gamma_conv1, 48*4)
S3s_b1_bn_beta_conv1 = readbinfile("./data/stage3_0_branch1_3_bias.bin", 48)
xlnk.cma_memcopy(s3s_b1_bn_beta_conv1, S3s_b1_bn_beta_conv1, 48*4)

S3s_b2_w_conv1 = readbinfile("./data/stage3_0_branch2_0_weight.bin", 48*48*1*1)
S3s_b2_w_conv1 = S3s_b2_w_conv1.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3s_b2_w_conv1, S3s_b2_w_conv1, 48*48*1*1*4)
S3s_b2_bn_mean_conv1 = readbinfile("./data/stage3_0_branch2_1_running_mean.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_mean_conv1, S3s_b2_bn_mean_conv1, 48*4)
S3s_b2_bn_val_conv1 = readbinfile("./data/stage3_0_branch2_1_running_var.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_val_conv1, S3s_b2_bn_val_conv1, 48*4)
S3s_b2_bn_gamma_conv1 = readbinfile("./data/stage3_0_branch2_1_weight.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_gamma_conv1, S3s_b2_bn_gamma_conv1, 48*4)
S3s_b2_bn_beta_conv1 = readbinfile("./data/stage3_0_branch2_1_bias.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_beta_conv1, S3s_b2_bn_beta_conv1, 48*4)

S3s_b2_w_convdw = readbinfile("./data/stage3_0_branch2_3_weight.bin", 48*3*3)
S3s_b2_w_convdw = S3s_b2_w_convdw.reshape((48, 3, 3))
xlnk.cma_memcopy(s3s_b2_w_convdw, S3s_b2_w_convdw, 48*3*3*4)
S3s_b2_bn_mean_convdw = readbinfile("./data/stage3_0_branch2_4_running_mean.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_mean_convdw, S3s_b2_bn_mean_convdw, 48*4)
S3s_b2_bn_val_convdw = readbinfile("./data/stage3_0_branch2_4_running_var.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_val_convdw, S3s_b2_bn_val_convdw, 48*4)
S3s_b2_bn_gamma_convdw = readbinfile("./data/stage3_0_branch2_4_weight.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_gamma_convdw, S3s_b2_bn_gamma_convdw, 48*4)
S3s_b2_bn_beta_convdw = readbinfile("./data/stage3_0_branch2_4_bias.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_beta_convdw, S3s_b2_bn_beta_convdw, 48*4)

S3s_b2_w_conv2 = readbinfile("./data/stage3_0_branch2_5_weight.bin", 48*48*1*1)
S3s_b2_w_conv2 = S3s_b2_w_conv2.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3s_b2_w_conv2, S3s_b2_w_conv2, 48*48*1*1*4)
S3s_b2_bn_mean_conv2 = readbinfile("./data/stage3_0_branch2_6_running_mean.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_mean_conv2, S3s_b2_bn_mean_conv2, 48*4)
S3s_b2_bn_val_conv2 = readbinfile("./data/stage3_0_branch2_6_running_var.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_val_conv2, S3s_b2_bn_val_conv2, 48*4)
S3s_b2_bn_gamma_conv2 = readbinfile("./data/stage3_0_branch2_6_weight.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_gamma_conv2, S3s_b2_bn_gamma_conv2, 48*4)
S3s_b2_bn_beta_conv2 = readbinfile("./data/stage3_0_branch2_6_bias.bin", 48)
xlnk.cma_memcopy(s3s_b2_bn_beta_conv2, S3s_b2_bn_beta_conv2, 48*4)


S3c1_b2_w_conv1 = readbinfile("./data/stage3_1_branch2_0_weight.bin", 48*48*1*1)
S3c1_b2_w_conv1 = S3c1_b2_w_conv1.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c1_b2_w_conv1, S3c1_b2_w_conv1, 48*48*1*1*4)
S3c1_b2_bn_mean_conv1 = readbinfile("./data/stage3_1_branch2_1_running_mean.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_mean_conv1, S3c1_b2_bn_mean_conv1, 48*4)
S3c1_b2_bn_val_conv1 = readbinfile("./data/stage3_1_branch2_1_running_var.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_val_conv1, S3c1_b2_bn_val_conv1, 48*4)
S3c1_b2_bn_gamma_conv1 = readbinfile("./data/stage3_1_branch2_1_weight.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_gamma_conv1, S3c1_b2_bn_gamma_conv1, 48*4)
S3c1_b2_bn_beta_conv1 = readbinfile("./data/stage3_1_branch2_1_bias.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_beta_conv1, S3c1_b2_bn_beta_conv1, 48*4)

S3c1_b2_w_convdw = readbinfile("./data/stage3_1_branch2_3_weight.bin", 48*3*3)
S3c1_b2_w_convdw = S3c1_b2_w_convdw.reshape((48, 3, 3))
xlnk.cma_memcopy(s3c1_b2_w_convdw, S3c1_b2_w_convdw, 48*3*3*4)
S3c1_b2_bn_mean_convdw = readbinfile("./data/stage3_1_branch2_4_running_mean.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_mean_convdw, S3c1_b2_bn_mean_convdw, 48*4)
S3c1_b2_bn_val_convdw = readbinfile("./data/stage3_1_branch2_4_running_var.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_val_convdw, S3c1_b2_bn_val_convdw, 48*4)
S3c1_b2_bn_gamma_convdw = readbinfile("./data/stage3_1_branch2_4_weight.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_gamma_convdw, S3c1_b2_bn_gamma_convdw, 48*4)
S3c1_b2_bn_beta_convdw = readbinfile("./data/stage3_1_branch2_4_bias.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_beta_convdw, S3c1_b2_bn_beta_convdw, 48*4)

S3c1_b2_w_conv2 = readbinfile("./data/stage3_1_branch2_5_weight.bin", 48*48*1*1)
S3c1_b2_w_conv2 = S3c1_b2_w_conv2.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c1_b2_w_conv2, S3c1_b2_w_conv2, 48*48*1*1*4)
S3c1_b2_bn_mean_conv2 = readbinfile("./data/stage3_1_branch2_6_running_mean.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_mean_conv2, S3c1_b2_bn_mean_conv2, 48*4)
S3c1_b2_bn_val_conv2 = readbinfile("./data/stage3_1_branch2_6_running_var.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_val_conv2, S3c1_b2_bn_val_conv2, 48*4)
S3c1_b2_bn_gamma_conv2 = readbinfile("./data/stage3_1_branch2_6_weight.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_gamma_conv2, S3c1_b2_bn_gamma_conv2, 48*4)
S3c1_b2_bn_beta_conv2 = readbinfile("./data/stage3_1_branch2_6_bias.bin", 48)
xlnk.cma_memcopy(s3c1_b2_bn_beta_conv2, S3c1_b2_bn_beta_conv2, 48*4)


S3c2_b2_w_conv1 = readbinfile("./data/stage3_2_branch2_0_weight.bin", 48*48*1*1)
S3c2_b2_w_conv1 = S3c2_b2_w_conv1.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c2_b2_w_conv1, S3c2_b2_w_conv1, 48*48*1*1*4)
S3c2_b2_bn_mean_conv1 = readbinfile("./data/stage3_2_branch2_1_running_mean.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_mean_conv1, S3c2_b2_bn_mean_conv1, 48*4)
S3c2_b2_bn_val_conv1 = readbinfile("./data/stage3_2_branch2_1_running_var.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_val_conv1, S3c2_b2_bn_val_conv1, 48*4)
S3c2_b2_bn_gamma_conv1 = readbinfile("./data/stage3_2_branch2_1_weight.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_gamma_conv1, S3c2_b2_bn_gamma_conv1, 48*4)
S3c2_b2_bn_beta_conv1 = readbinfile("./data/stage3_2_branch2_1_bias.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_beta_conv1, S3c2_b2_bn_beta_conv1, 48*4)

S3c2_b2_w_convdw = readbinfile("./data/stage3_2_branch2_3_weight.bin", 48*3*3)
S3c2_b2_w_convdw = S3c2_b2_w_convdw.reshape((48, 3, 3))
xlnk.cma_memcopy(s3c2_b2_w_convdw, S3c2_b2_w_convdw, 48*3*3*4)
S3c2_b2_bn_mean_convdw = readbinfile("./data/stage3_2_branch2_4_running_mean.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_mean_convdw, S3c2_b2_bn_mean_convdw, 48*4)
S3c2_b2_bn_val_convdw = readbinfile("./data/stage3_2_branch2_4_running_var.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_val_convdw, S3c2_b2_bn_val_convdw, 48*4)
S3c2_b2_bn_gamma_convdw = readbinfile("./data/stage3_2_branch2_4_weight.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_gamma_convdw, S3c2_b2_bn_gamma_convdw, 48*4)
S3c2_b2_bn_beta_convdw = readbinfile("./data/stage3_2_branch2_4_bias.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_beta_convdw, S3c2_b2_bn_beta_convdw, 48*4)

S3c2_b2_w_conv2 = readbinfile("./data/stage3_2_branch2_5_weight.bin", 48*48*1*1)
S3c2_b2_w_conv2 = S3c2_b2_w_conv2.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c2_b2_w_conv2, S3c2_b2_w_conv2, 48*48*1*1*4)
S3c2_b2_bn_mean_conv2 = readbinfile("./data/stage3_2_branch2_6_running_mean.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_mean_conv2, S3c2_b2_bn_mean_conv2, 48*4)
S3c2_b2_bn_val_conv2 = readbinfile("./data/stage3_2_branch2_6_running_var.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_val_conv2, S3c2_b2_bn_val_conv2, 48*4)
S3c2_b2_bn_gamma_conv2 = readbinfile("./data/stage3_2_branch2_6_weight.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_gamma_conv2, S3c2_b2_bn_gamma_conv2, 48*4)
S3c2_b2_bn_beta_conv2 = readbinfile("./data/stage3_2_branch2_6_bias.bin", 48)
xlnk.cma_memcopy(s3c2_b2_bn_beta_conv2, S3c2_b2_bn_beta_conv2, 48*4)


S3c3_b2_w_conv1 = readbinfile("./data/stage3_3_branch2_0_weight.bin", 48*48*1*1)
S3c3_b2_w_conv1 = S3c3_b2_w_conv1.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c3_b2_w_conv1, S3c3_b2_w_conv1, 48*48*1*1*4)
S3c3_b2_bn_mean_conv1 = readbinfile("./data/stage3_3_branch2_1_running_mean.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_mean_conv1, S3c3_b2_bn_mean_conv1, 48*4)
S3c3_b2_bn_val_conv1 = readbinfile("./data/stage3_3_branch2_1_running_var.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_val_conv1, S3c3_b2_bn_val_conv1, 48*4)
S3c3_b2_bn_gamma_conv1 = readbinfile("./data/stage3_3_branch2_1_weight.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_gamma_conv1, S3c3_b2_bn_gamma_conv1, 48*4)
S3c3_b2_bn_beta_conv1 = readbinfile("./data/stage3_3_branch2_1_bias.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_beta_conv1, S3c3_b2_bn_beta_conv1, 48*4)

S3c3_b2_w_convdw = readbinfile("./data/stage3_3_branch2_3_weight.bin", 48*3*3)
S3c3_b2_w_convdw = S3c3_b2_w_convdw.reshape((48, 3, 3))
xlnk.cma_memcopy(s3c3_b2_w_convdw, S3c3_b2_w_convdw, 48*3*3*4)
S3c3_b2_bn_mean_convdw = readbinfile("./data/stage3_3_branch2_4_running_mean.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_mean_convdw, S3c3_b2_bn_mean_convdw, 48*4)
S3c3_b2_bn_val_convdw = readbinfile("./data/stage3_3_branch2_4_running_var.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_val_convdw, S3c3_b2_bn_val_convdw, 48*4)
S3c3_b2_bn_gamma_convdw = readbinfile("./data/stage3_3_branch2_4_weight.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_gamma_convdw, S3c3_b2_bn_gamma_convdw, 48*4)
S3c3_b2_bn_beta_convdw = readbinfile("./data/stage3_3_branch2_4_bias.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_beta_convdw, S3c3_b2_bn_beta_convdw, 48*4)

S3c3_b2_w_conv2 = readbinfile("./data/stage3_3_branch2_5_weight.bin", 48*48*1*1)
S3c3_b2_w_conv2 = S3c3_b2_w_conv2.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c3_b2_w_conv2, S3c3_b2_w_conv2, 48*48*1*1*4)
S3c3_b2_bn_mean_conv2 = readbinfile("./data/stage3_3_branch2_6_running_mean.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_mean_conv2, S3c3_b2_bn_mean_conv2, 48*4)
S3c3_b2_bn_val_conv2 = readbinfile("./data/stage3_3_branch2_6_running_var.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_val_conv2, S3c3_b2_bn_val_conv2, 48*4)
S3c3_b2_bn_gamma_conv2 = readbinfile("./data/stage3_3_branch2_6_weight.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_gamma_conv2, S3c3_b2_bn_gamma_conv2, 48*4)
S3c3_b2_bn_beta_conv2 = readbinfile("./data/stage3_3_branch2_6_bias.bin", 48)
xlnk.cma_memcopy(s3c3_b2_bn_beta_conv2, S3c3_b2_bn_beta_conv2, 48*4)


S3c4_b2_w_conv1 = readbinfile("./data/stage3_4_branch2_0_weight.bin", 48*48*1*1)
S3c4_b2_w_conv1 = S3c4_b2_w_conv1.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c4_b2_w_conv1, S3c4_b2_w_conv1, 48*48*1*1*4)
S3c4_b2_bn_mean_conv1 = readbinfile("./data/stage3_4_branch2_1_running_mean.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_mean_conv1, S3c4_b2_bn_mean_conv1, 48*4)
S3c4_b2_bn_val_conv1 = readbinfile("./data/stage3_4_branch2_1_running_var.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_val_conv1, S3c4_b2_bn_val_conv1, 48*4)
S3c4_b2_bn_gamma_conv1 = readbinfile("./data/stage3_4_branch2_1_weight.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_gamma_conv1, S3c4_b2_bn_gamma_conv1, 48*4)
S3c4_b2_bn_beta_conv1 = readbinfile("./data/stage3_4_branch2_1_bias.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_beta_conv1, S3c4_b2_bn_beta_conv1, 48*4)

S3c4_b2_w_convdw = readbinfile("./data/stage3_4_branch2_3_weight.bin", 48*3*3)
S3c4_b2_w_convdw = S3c4_b2_w_convdw.reshape((48, 3, 3))
xlnk.cma_memcopy(s3c4_b2_w_convdw, S3c4_b2_w_convdw, 48*3*3*4)
S3c4_b2_bn_mean_convdw = readbinfile("./data/stage3_4_branch2_4_running_mean.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_mean_convdw, S3c4_b2_bn_mean_convdw, 48*4)
S3c4_b2_bn_val_convdw = readbinfile("./data/stage3_4_branch2_4_running_var.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_val_convdw, S3c4_b2_bn_val_convdw, 48*4)
S3c4_b2_bn_gamma_convdw = readbinfile("./data/stage3_4_branch2_4_weight.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_gamma_convdw, S3c4_b2_bn_gamma_convdw, 48*4)
S3c4_b2_bn_beta_convdw = readbinfile("./data/stage3_4_branch2_4_bias.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_beta_convdw, S3c4_b2_bn_beta_convdw, 48*4)

S3c4_b2_w_conv2 = readbinfile("./data/stage3_4_branch2_5_weight.bin", 48*48*1*1)
S3c4_b2_w_conv2 = S3c4_b2_w_conv2.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c4_b2_w_conv2, S3c4_b2_w_conv2, 48*48*1*1*4)
S3c4_b2_bn_mean_conv2 = readbinfile("./data/stage3_4_branch2_6_running_mean.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_mean_conv2, S3c4_b2_bn_mean_conv2, 48*4)
S3c4_b2_bn_val_conv2 = readbinfile("./data/stage3_4_branch2_6_running_var.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_val_conv2, S3c4_b2_bn_val_conv2, 48*4)
S3c4_b2_bn_gamma_conv2 = readbinfile("./data/stage3_4_branch2_6_weight.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_gamma_conv2, S3c4_b2_bn_gamma_conv2, 48*4)
S3c4_b2_bn_beta_conv2 = readbinfile("./data/stage3_4_branch2_6_bias.bin", 48)
xlnk.cma_memcopy(s3c4_b2_bn_beta_conv2, S3c4_b2_bn_beta_conv2, 48*4)


S3c5_b2_w_conv1 = readbinfile("./data/stage3_5_branch2_0_weight.bin", 48*48*1*1)
S3c5_b2_w_conv1 = S3c5_b2_w_conv1.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c5_b2_w_conv1, S3c5_b2_w_conv1, 48*48*1*1*4)
S3c5_b2_bn_mean_conv1 = readbinfile("./data/stage3_5_branch2_1_running_mean.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_mean_conv1, S3c5_b2_bn_mean_conv1, 48*4)
S3c5_b2_bn_val_conv1 = readbinfile("./data/stage3_5_branch2_1_running_var.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_val_conv1, S3c5_b2_bn_val_conv1, 48*4)
S3c5_b2_bn_gamma_conv1 = readbinfile("./data/stage3_5_branch2_1_weight.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_gamma_conv1, S3c5_b2_bn_gamma_conv1, 48*4)
S3c5_b2_bn_beta_conv1 = readbinfile("./data/stage3_5_branch2_1_bias.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_beta_conv1, S3c5_b2_bn_beta_conv1, 48*4)

S3c5_b2_w_convdw = readbinfile("./data/stage3_5_branch2_3_weight.bin", 48*3*3)
S3c5_b2_w_convdw = S3c5_b2_w_convdw.reshape((48, 3, 3))
xlnk.cma_memcopy(s3c5_b2_w_convdw, S3c5_b2_w_convdw, 48*3*3*4)
S3c5_b2_bn_mean_convdw = readbinfile("./data/stage3_5_branch2_4_running_mean.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_mean_convdw, S3c5_b2_bn_mean_convdw, 48*4)
S3c5_b2_bn_val_convdw = readbinfile("./data/stage3_5_branch2_4_running_var.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_val_convdw, S3c5_b2_bn_val_convdw, 48*4)
S3c5_b2_bn_gamma_convdw = readbinfile("./data/stage3_5_branch2_4_weight.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_gamma_convdw, S3c5_b2_bn_gamma_convdw, 48*4)
S3c5_b2_bn_beta_convdw = readbinfile("./data/stage3_5_branch2_4_bias.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_beta_convdw, S3c5_b2_bn_beta_convdw, 48*4)

S3c5_b2_w_conv2 = readbinfile("./data/stage3_5_branch2_5_weight.bin", 48*48*1*1)
S3c5_b2_w_conv2 = S3c5_b2_w_conv2.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c5_b2_w_conv2, S3c5_b2_w_conv2, 48*48*1*1*4)
S3c5_b2_bn_mean_conv2 = readbinfile("./data/stage3_5_branch2_6_running_mean.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_mean_conv2, S3c5_b2_bn_mean_conv2, 48*4)
S3c5_b2_bn_val_conv2 = readbinfile("./data/stage3_5_branch2_6_running_var.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_val_conv2, S3c5_b2_bn_val_conv2, 48*4)
S3c5_b2_bn_gamma_conv2 = readbinfile("./data/stage3_5_branch2_6_weight.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_gamma_conv2, S3c5_b2_bn_gamma_conv2, 48*4)
S3c5_b2_bn_beta_conv2 = readbinfile("./data/stage3_5_branch2_6_bias.bin", 48)
xlnk.cma_memcopy(s3c5_b2_bn_beta_conv2, S3c5_b2_bn_beta_conv2, 48*4)


S3c6_b2_w_conv1 = readbinfile("./data/stage3_6_branch2_0_weight.bin", 48*48*1*1)
S3c6_b2_w_conv1 = S3c6_b2_w_conv1.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c6_b2_w_conv1, S3c6_b2_w_conv1, 48*48*1*1*4)
S3c6_b2_bn_mean_conv1 = readbinfile("./data/stage3_6_branch2_1_running_mean.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_mean_conv1, S3c6_b2_bn_mean_conv1, 48*4)
S3c6_b2_bn_val_conv1 = readbinfile("./data/stage3_6_branch2_1_running_var.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_val_conv1, S3c6_b2_bn_val_conv1, 48*4)
S3c6_b2_bn_gamma_conv1 = readbinfile("./data/stage3_6_branch2_1_weight.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_gamma_conv1, S3c6_b2_bn_gamma_conv1, 48*4)
S3c6_b2_bn_beta_conv1 = readbinfile("./data/stage3_6_branch2_1_bias.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_beta_conv1, S3c6_b2_bn_beta_conv1, 48*4)

S3c6_b2_w_convdw = readbinfile("./data/stage3_6_branch2_3_weight.bin", 48*3*3)
S3c6_b2_w_convdw = S3c6_b2_w_convdw.reshape((48, 3, 3))
xlnk.cma_memcopy(s3c6_b2_w_convdw, S3c6_b2_w_convdw, 48*3*3*4)
S3c6_b2_bn_mean_convdw = readbinfile("./data/stage3_6_branch2_4_running_mean.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_mean_convdw, S3c6_b2_bn_mean_convdw, 48*4)
S3c6_b2_bn_val_convdw = readbinfile("./data/stage3_6_branch2_4_running_var.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_val_convdw, S3c6_b2_bn_val_convdw, 48*4)
S3c6_b2_bn_gamma_convdw = readbinfile("./data/stage3_6_branch2_4_weight.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_gamma_convdw, S3c6_b2_bn_gamma_convdw, 48*4)
S3c6_b2_bn_beta_convdw = readbinfile("./data/stage3_6_branch2_4_bias.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_beta_convdw, S3c6_b2_bn_beta_convdw, 48*4)

S3c6_b2_w_conv2 = readbinfile("./data/stage3_6_branch2_5_weight.bin", 48*48*1*1)
S3c6_b2_w_conv2 = S3c6_b2_w_conv2.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c6_b2_w_conv2, S3c6_b2_w_conv2, 48*48*1*1*4)
S3c6_b2_bn_mean_conv2 = readbinfile("./data/stage3_6_branch2_6_running_mean.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_mean_conv2, S3c6_b2_bn_mean_conv2, 48*4)
S3c6_b2_bn_val_conv2 = readbinfile("./data/stage3_6_branch2_6_running_var.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_val_conv2, S3c6_b2_bn_val_conv2, 48*4)
S3c6_b2_bn_gamma_conv2 = readbinfile("./data/stage3_6_branch2_6_weight.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_gamma_conv2, S3c6_b2_bn_gamma_conv2, 48*4)
S3c6_b2_bn_beta_conv2 = readbinfile("./data/stage3_6_branch2_6_bias.bin", 48)
xlnk.cma_memcopy(s3c6_b2_bn_beta_conv2, S3c6_b2_bn_beta_conv2, 48*4)


S3c7_b2_w_conv1 = readbinfile("./data/stage3_7_branch2_0_weight.bin", 48*48*1*1)
S3c7_b2_w_conv1 = S3c7_b2_w_conv1.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c7_b2_w_conv1, S3c7_b2_w_conv1, 48*48*1*1*4)
S3c7_b2_bn_mean_conv1 = readbinfile("./data/stage3_7_branch2_1_running_mean.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_mean_conv1, S3c7_b2_bn_mean_conv1, 48*4)
S3c7_b2_bn_val_conv1 = readbinfile("./data/stage3_7_branch2_1_running_var.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_val_conv1, S3c7_b2_bn_val_conv1, 48*4)
S3c7_b2_bn_gamma_conv1 = readbinfile("./data/stage3_7_branch2_1_weight.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_gamma_conv1, S3c7_b2_bn_gamma_conv1, 48*4)
S3c7_b2_bn_beta_conv1 = readbinfile("./data/stage3_7_branch2_1_bias.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_beta_conv1, S3c7_b2_bn_beta_conv1, 48*4)

S3c7_b2_w_convdw = readbinfile("./data/stage3_7_branch2_3_weight.bin", 48*3*3)
S3c7_b2_w_convdw = S3c7_b2_w_convdw.reshape((48, 3, 3))
xlnk.cma_memcopy(s3c7_b2_w_convdw, S3c7_b2_w_convdw, 48*3*3*4)
S3c7_b2_bn_mean_convdw = readbinfile("./data/stage3_7_branch2_4_running_mean.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_mean_convdw, S3c7_b2_bn_mean_convdw, 48*4)
S3c7_b2_bn_val_convdw = readbinfile("./data/stage3_7_branch2_4_running_var.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_val_convdw, S3c7_b2_bn_val_convdw, 48*4)
S3c7_b2_bn_gamma_convdw = readbinfile("./data/stage3_7_branch2_4_weight.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_gamma_convdw, S3c7_b2_bn_gamma_convdw, 48*4)
S3c7_b2_bn_beta_convdw = readbinfile("./data/stage3_7_branch2_4_bias.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_beta_convdw, S3c7_b2_bn_beta_convdw, 48*4)

S3c7_b2_w_conv2 = readbinfile("./data/stage3_7_branch2_5_weight.bin", 48*48*1*1)
S3c7_b2_w_conv2 = S3c7_b2_w_conv2.reshape((48, 48, 1, 1))
xlnk.cma_memcopy(s3c7_b2_w_conv2, S3c7_b2_w_conv2, 48*48*1*1*4)
S3c7_b2_bn_mean_conv2 = readbinfile("./data/stage3_7_branch2_6_running_mean.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_mean_conv2, S3c7_b2_bn_mean_conv2, 48*4)
S3c7_b2_bn_val_conv2 = readbinfile("./data/stage3_7_branch2_6_running_var.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_val_conv2, S3c7_b2_bn_val_conv2, 48*4)
S3c7_b2_bn_gamma_conv2 = readbinfile("./data/stage3_7_branch2_6_weight.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_gamma_conv2, S3c7_b2_bn_gamma_conv2, 48*4)
S3c7_b2_bn_beta_conv2 = readbinfile("./data/stage3_7_branch2_6_bias.bin", 48)
xlnk.cma_memcopy(s3c7_b2_bn_beta_conv2, S3c7_b2_bn_beta_conv2, 48*4)

print("successfully!")


successfully!


 ## <span style="font-size: 24px;">***stage 4 代码块（核心部分）***</span>

In [14]:
'''                                          stage4                                                  '''
#stage4下采样单元-分支1-3x3深度卷积输出
out_s4s_b1_convdw = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4下采样单元-分支1-3x3深度卷积核，BN参数
s4s_b1_w_convdw = xlnk.cma_array(shape = (96, 3, 3), cacheable = 0, dtype = np.float32)
S4s_b1_w_convdw = readbinfile("./data/stage4_0_branch1_0_weight.bin", 96*3*3)
S4s_b1_w_convdw = S4s_b1_w_convdw.reshape((96, 3, 3))
xlnk.cma_memcopy(s4s_b1_w_convdw, S4s_b1_w_convdw, 96*3*3*4)

s4s_b1_bn_mean_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b1_bn_mean_convdw = readbinfile("./data/stage4_0_branch1_1_running_mean.bin", 96)
S4s_b1_bn_mean_convdw = S4s_b1_bn_mean_convdw.reshape((96))
xlnk.cma_memcopy(s4s_b1_bn_mean_convdw, S4s_b1_bn_mean_convdw, 96*4)

s4s_b1_bn_val_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b1_bn_val_convdw = readbinfile("./data/stage4_0_branch1_1_running_var.bin", 96)
S4s_b1_bn_val_convdw = S4s_b1_bn_val_convdw.reshape((96))
xlnk.cma_memcopy(s4s_b1_bn_val_convdw, S4s_b1_bn_val_convdw, 96*4)

s4s_b1_bn_gamma_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b1_bn_gamma_convdw = readbinfile("./data/stage4_0_branch1_1_weight.bin", 96)
S4s_b1_bn_gamma_convdw = S4s_b1_bn_gamma_convdw.reshape((96))
xlnk.cma_memcopy(s4s_b1_bn_gamma_convdw, S4s_b1_bn_gamma_convdw, 96*4)

s4s_b1_bn_beta_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b1_bn_beta_convdw = readbinfile("./data/stage4_0_branch1_1_bias.bin", 96)
S4s_b1_bn_beta_convdw = S4s_b1_bn_beta_convdw.reshape((96))
xlnk.cma_memcopy(s4s_b1_bn_beta_convdw, S4s_b1_bn_beta_convdw, 96*4)

#stage4下采样单元-分支1-1x1普通卷积输出
out_s4s_b1_conv1 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4下采样单元-分支1-1x1普通卷积核，BN参数
s4s_b1_w_conv1 = xlnk.cma_array(shape = (96, 96, 1, 1), cacheable = 0, dtype = np.float32)
S4s_b1_w_conv1 = readbinfile("./data/stage4_0_branch1_2_weight.bin", 96*96*1*1)
S4s_b1_w_conv1 = S4s_b1_w_conv1.reshape((96, 96, 1, 1))
xlnk.cma_memcopy(s4s_b1_w_conv1, S4s_b1_w_conv1, 96*96*1*1*4)

s4s_b1_bn_mean_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b1_bn_mean_conv1 = readbinfile("./data/stage4_0_branch1_3_running_mean.bin", 96)
S4s_b1_bn_mean_conv1 = S4s_b1_bn_mean_conv1.reshape((96))
xlnk.cma_memcopy(s4s_b1_bn_mean_conv1, S4s_b1_bn_mean_conv1, 96*4)

s4s_b1_bn_val_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b1_bn_val_conv1 = readbinfile("./data/stage4_0_branch1_3_running_var.bin", 96)
S4s_b1_bn_val_conv1 = S4s_b1_bn_val_conv1.reshape((96))
xlnk.cma_memcopy(s4s_b1_bn_val_conv1, S4s_b1_bn_val_conv1, 96*4)

s4s_b1_bn_gamma_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b1_bn_gamma_conv1 = readbinfile("./data/stage4_0_branch1_3_weight.bin", 96)
S4s_b1_bn_gamma_conv1 = S4s_b1_bn_gamma_conv1.reshape((96))
xlnk.cma_memcopy(s4s_b1_bn_gamma_conv1, S4s_b1_bn_gamma_conv1, 96*4)

s4s_b1_bn_beta_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b1_bn_beta_conv1 = readbinfile("./data/stage4_0_branch1_3_bias.bin", 96)
S4s_b1_bn_beta_conv1 = S4s_b1_bn_beta_conv1.reshape((96))
xlnk.cma_memcopy(s4s_b1_bn_beta_conv1, S4s_b1_bn_beta_conv1, 96*4)

#stage4下采样单元-分支2-1x1普通卷积1输出
out_s4s_b2_conv1 = xlnk.cma_array(shape = (8, 8, 96), cacheable = 0, dtype = np.float32)
#stage4下采样单元-分支2-1x1普通卷积1核，BN参数
s4s_b2_w_conv1 = xlnk.cma_array(shape = (96, 96, 1, 1), cacheable = 0, dtype = np.float32)
S4s_b2_w_conv1 = readbinfile("./data/stage4_0_branch2_0_weight.bin", 96*96*1*1)
S4s_b2_w_conv1 = S4s_b2_w_conv1.reshape((96, 96, 1, 1))
xlnk.cma_memcopy(s4s_b2_w_conv1, S4s_b2_w_conv1, 96*96*1*1*4)

s4s_b2_bn_mean_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_mean_conv1 = readbinfile("./data/stage4_0_branch2_1_running_mean.bin", 96)
S4s_b2_bn_mean_conv1 = S4s_b2_bn_mean_conv1.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_mean_conv1, S4s_b2_bn_mean_conv1, 96*4)

s4s_b2_bn_val_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_val_conv1 = readbinfile("./data/stage4_0_branch2_1_running_var.bin", 96)
S4s_b2_bn_val_conv1 = S4s_b2_bn_val_conv1.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_val_conv1, S4s_b2_bn_val_conv1, 96*4)

s4s_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_gamma_conv1 = readbinfile("./data/stage4_0_branch2_1_weight.bin", 96)
S4s_b2_bn_gamma_conv1 = S4s_b2_bn_gamma_conv1.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_gamma_conv1, S4s_b2_bn_gamma_conv1, 96*4)

s4s_b2_bn_beta_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_beta_conv1 = readbinfile("./data/stage4_0_branch2_1_bias.bin", 96)
S4s_b2_bn_beta_conv1 = S4s_b2_bn_beta_conv1.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_beta_conv1, S4s_b2_bn_beta_conv1, 96*4)

#stage4下采样单元-分支2-3x3深度卷积输出
out_s4s_b2_convdw = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4下采样单元-分支2-3x3深度卷积核，BN参数
s4s_b2_w_convdw = xlnk.cma_array(shape = (96, 3, 3), cacheable = 0, dtype = np.float32)
S4s_b2_w_convdw = readbinfile("./data/stage4_0_branch2_3_weight.bin", 96*3*3)
S4s_b2_w_convdw = S4s_b2_w_convdw.reshape((96, 3, 3))
xlnk.cma_memcopy(s4s_b2_w_convdw, S4s_b2_w_convdw, 96*3*3*4)

s4s_b2_bn_mean_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_mean_convdw = readbinfile("./data/stage4_0_branch2_4_running_mean.bin", 96)
S4s_b2_bn_mean_convdw = S4s_b2_bn_mean_convdw.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_mean_convdw, S4s_b2_bn_mean_convdw, 96*4)

s4s_b2_bn_val_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_val_convdw = readbinfile("./data/stage4_0_branch2_4_running_var.bin", 96)
S4s_b2_bn_val_convdw = S4s_b2_bn_val_convdw.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_val_convdw, S4s_b2_bn_val_convdw, 96*4)

s4s_b2_bn_gamma_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_gamma_convdw = readbinfile("./data/stage4_0_branch2_4_weight.bin", 96)
S4s_b2_bn_gamma_convdw = S4s_b2_bn_gamma_convdw.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_gamma_convdw, S4s_b2_bn_gamma_convdw, 96*4)

s4s_b2_bn_beta_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_beta_convdw = readbinfile("./data/stage4_0_branch2_4_bias.bin", 96)
S4s_b2_bn_beta_convdw = S4s_b2_bn_beta_convdw.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_beta_convdw, S4s_b2_bn_beta_convdw, 96*4)

#stage4下采样单元-分支2-1x1普通卷积输出2
out_s4s_b2_conv2 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4下采样单元-分支2-1x1普通卷积核2，BN参数
s4s_b2_w_conv2 = xlnk.cma_array(shape = (96, 96, 1, 1), cacheable = 0, dtype = np.float32)
S4s_b2_w_conv2 = readbinfile("./data/stage4_0_branch2_5_weight.bin", 96*96*1*1)
S4s_b2_w_conv2 = S4s_b2_w_conv2.reshape((96, 96, 1, 1))
xlnk.cma_memcopy(s4s_b2_w_conv2, S4s_b2_w_conv2, 96*96*1*1*4)

s4s_b2_bn_mean_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_mean_conv2 = readbinfile("./data/stage4_0_branch2_6_running_mean.bin", 96)
S4s_b2_bn_mean_conv2 = S4s_b2_bn_mean_conv2.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_mean_conv2, S4s_b2_bn_mean_conv2, 96*4)

s4s_b2_bn_val_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_val_conv2 = readbinfile("./data/stage4_0_branch2_6_running_var.bin", 96)
S4s_b2_bn_val_conv2 = S4s_b2_bn_val_conv2.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_val_conv2, S4s_b2_bn_val_conv2, 96*4)

s4s_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_gamma_conv2 = readbinfile("./data/stage4_0_branch2_6_weight.bin", 96)
S4s_b2_bn_gamma_conv2 = S4s_b2_bn_gamma_conv2.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_gamma_conv2, S4s_b2_bn_gamma_conv2, 96*4)

s4s_b2_bn_beta_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4s_b2_bn_beta_conv2 = readbinfile("./data/stage4_0_branch2_6_bias.bin", 96)
S4s_b2_bn_beta_conv2 = S4s_b2_bn_beta_conv2.reshape((96))
xlnk.cma_memcopy(s4s_b2_bn_beta_conv2, S4s_b2_bn_beta_conv2, 96*4)

#stage4下采样单元-通道合并，通道混洗
in_s4s_shuff = xlnk.cma_array(shape = (4, 4, 192), cacheable = 0, dtype = np.float32)
out_s4s_shuff = xlnk.cma_array(shape = (4, 4, 192), cacheable = 0, dtype = np.float32)


#基本单元1
#stage4基本单元1-通道拆分
out_s4c1_ch_spilt1 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
out_s4c1_ch_spilt2 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)

#stage4基本单元1-分支1

#stage4基本单元1-分支2-1x1普通卷积1输出
out_s4c1_b2_conv1 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4基本单元1-分支2-1x1普通卷积1核，BN参数
s4c1_b2_w_conv1 = xlnk.cma_array(shape = (96, 96, 1, 1), cacheable = 0, dtype = np.float32)
S4c1_b2_w_conv1 = readbinfile("./data/stage4_1_branch2_0_weight.bin", 96*96*1*1)
S4c1_b2_w_conv1 = S4c1_b2_w_conv1.reshape((96, 96, 1, 1))
xlnk.cma_memcopy(s4c1_b2_w_conv1, S4c1_b2_w_conv1, 96*96*1*1*4)

s4c1_b2_bn_mean_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_mean_conv1 = readbinfile("./data/stage4_1_branch2_1_running_mean.bin", 96)
S4c1_b2_bn_mean_conv1 = S4c1_b2_bn_mean_conv1.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_mean_conv1, S4c1_b2_bn_mean_conv1, 96*4)

s4c1_b2_bn_val_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_val_conv1 = readbinfile("./data/stage4_1_branch2_1_running_var.bin", 96)
S4c1_b2_bn_val_conv1 = S4c1_b2_bn_val_conv1.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_val_conv1, S4c1_b2_bn_val_conv1, 96*4)

s4c1_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_gamma_conv1 = readbinfile("./data/stage4_1_branch2_1_weight.bin", 96)
S4c1_b2_bn_gamma_conv1 = S4c1_b2_bn_gamma_conv1.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_gamma_conv1, S4c1_b2_bn_gamma_conv1, 96*4)

s4c1_b2_bn_beta_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_beta_conv1 = readbinfile("./data/stage4_1_branch2_1_bias.bin", 96)
S4c1_b2_bn_beta_conv1 = S4c1_b2_bn_beta_conv1.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_beta_conv1, S4c1_b2_bn_beta_conv1, 96*4)

#stage4基本单元1-分支2-3x3深度卷积输出
out_s4c1_b2_convdw = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4基本单元1-分支2-3x3深度卷积核，BN参数
s4c1_b2_w_convdw = xlnk.cma_array(shape = (96, 3, 3), cacheable = 0, dtype = np.float32)
S4c1_b2_w_convdw = readbinfile("./data/stage4_1_branch2_3_weight.bin", 96*3*3)
S4c1_b2_w_convdw = S4c1_b2_w_convdw.reshape((96, 3, 3))
xlnk.cma_memcopy(s4c1_b2_w_convdw, S4c1_b2_w_convdw, 96*3*3*4)

s4c1_b2_bn_mean_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_mean_convdw = readbinfile("./data/stage4_1_branch2_4_running_mean.bin", 96)
S4c1_b2_bn_mean_convdw = S4c1_b2_bn_mean_convdw.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_mean_convdw, S4c1_b2_bn_mean_convdw, 96*4)

s4c1_b2_bn_val_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_val_convdw = readbinfile("./data/stage4_1_branch2_4_running_var.bin", 96)
S4c1_b2_bn_val_convdw = S4c1_b2_bn_val_convdw.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_val_convdw, S4c1_b2_bn_val_convdw, 96*4)

s4c1_b2_bn_gamma_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_gamma_convdw = readbinfile("./data/stage4_1_branch2_4_weight.bin", 96)
S4c1_b2_bn_gamma_convdw = S4c1_b2_bn_gamma_convdw.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_gamma_convdw, S4c1_b2_bn_gamma_convdw, 96*4)

s4c1_b2_bn_beta_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_beta_convdw = readbinfile("./data/stage4_1_branch2_4_bias.bin", 96)
S4c1_b2_bn_beta_convdw = S4c1_b2_bn_beta_convdw.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_beta_convdw, S4c1_b2_bn_beta_convdw, 96*4)

#stage4基本单元1-分支2-1x1普通卷积输出2
out_s4c1_b2_conv2 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4基本单元1-分支2-1x1普通卷积核2，BN参数
s4c1_b2_w_conv2 = xlnk.cma_array(shape = (96, 96, 1, 1), cacheable = 0, dtype = np.float32)
S4c1_b2_w_conv2 = readbinfile("./data/stage4_1_branch2_5_weight.bin", 96*96*1*1)
S4c1_b2_w_conv2 = S4c1_b2_w_conv2.reshape((96, 96, 1, 1))
xlnk.cma_memcopy(s4c1_b2_w_conv2, S4c1_b2_w_conv2, 96*96*1*1*4)

s4c1_b2_bn_mean_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_mean_conv2 = readbinfile("./data/stage4_1_branch2_6_running_mean.bin", 96)
S4c1_b2_bn_mean_conv2 = S4c1_b2_bn_mean_conv2.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_mean_conv2, S4c1_b2_bn_mean_conv2, 96*4)

s4c1_b2_bn_val_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_val_conv2 = readbinfile("./data/stage4_1_branch2_6_running_var.bin", 96)
S4c1_b2_bn_val_conv2 = S4c1_b2_bn_val_conv2.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_val_conv2, S4c1_b2_bn_val_conv2, 96*4)

s4c1_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_gamma_conv2 = readbinfile("./data/stage4_1_branch2_6_weight.bin", 96)
S4c1_b2_bn_gamma_conv2 = S4c1_b2_bn_gamma_conv2.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_gamma_conv2, S4c1_b2_bn_gamma_conv2, 96*4)

s4c1_b2_bn_beta_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c1_b2_bn_beta_conv2 = readbinfile("./data/stage4_1_branch2_6_bias.bin", 96)
S4c1_b2_bn_beta_conv2 = S4c1_b2_bn_beta_conv2.reshape((96))
xlnk.cma_memcopy(s4c1_b2_bn_beta_conv2, S4c1_b2_bn_beta_conv2, 96*4)

#stage4基本单元1-通道合并，通道混洗
in_s4c1_shuff = xlnk.cma_array(shape = (4, 4, 192), cacheable = 0, dtype = np.float32)
out_s4c1_shuff = xlnk.cma_array(shape = (4, 4, 192), cacheable = 0, dtype = np.float32)


#基本单元2
#stage4基本单元2-通道拆分
out_s4c2_ch_spilt1 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
out_s4c2_ch_spilt2 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)  

#stage4基本单元2-分支1

#stage4基本单元2-分支2-1x1普通卷积1输出
out_s4c2_b2_conv1 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4基本单元2-分支2-1x1普通卷积1核，BN参数
s4c2_b2_w_conv1 = xlnk.cma_array(shape = (96, 96, 1, 1), cacheable = 0, dtype = np.float32)
S4c2_b2_w_conv1 = readbinfile("./data/stage4_2_branch2_0_weight.bin", 96*96*1*1)
S4c2_b2_w_conv1 = S4c2_b2_w_conv1.reshape((96, 96, 1, 1))
xlnk.cma_memcopy(s4c2_b2_w_conv1, S4c2_b2_w_conv1, 96*96*1*1*4)

s4c2_b2_bn_mean_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_mean_conv1 = readbinfile("./data/stage4_2_branch2_1_running_mean.bin", 96)
S4c2_b2_bn_mean_conv1 = S4c2_b2_bn_mean_conv1.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_mean_conv1, S4c2_b2_bn_mean_conv1, 96*4)

s4c2_b2_bn_val_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_val_conv1 = readbinfile("./data/stage4_2_branch2_1_running_var.bin", 96)
S4c2_b2_bn_val_conv1 = S4c2_b2_bn_val_conv1.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_val_conv1, S4c2_b2_bn_val_conv1, 96*4)

s4c2_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_gamma_conv1 = readbinfile("./data/stage4_2_branch2_1_weight.bin", 96)
S4c2_b2_bn_gamma_conv1 = S4c2_b2_bn_gamma_conv1.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_gamma_conv1, S4c2_b2_bn_gamma_conv1, 96*4)

s4c2_b2_bn_beta_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_beta_conv1 = readbinfile("./data/stage4_2_branch2_1_bias.bin", 96)
S4c2_b2_bn_beta_conv1 = S4c2_b2_bn_beta_conv1.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_beta_conv1, S4c2_b2_bn_beta_conv1, 96*4)

#stage4基本单元2-分支2-3x3深度卷积输出
out_s4c2_b2_convdw = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4基本单元2-分支2-3x3深度卷积核，BN参数
s4c2_b2_w_convdw = xlnk.cma_array(shape = (96, 3, 3), cacheable = 0, dtype = np.float32)
S4c2_b2_w_convdw = readbinfile("./data/stage4_2_branch2_3_weight.bin", 96*3*3)
S4c2_b2_w_convdw = S4c2_b2_w_convdw.reshape((96, 3, 3))
xlnk.cma_memcopy(s4c2_b2_w_convdw, S4c2_b2_w_convdw, 96*3*3*4)

s4c2_b2_bn_mean_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_mean_convdw = readbinfile("./data/stage4_2_branch2_4_running_mean.bin", 96)
S4c2_b2_bn_mean_convdw = S4c2_b2_bn_mean_convdw.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_mean_convdw, S4c2_b2_bn_mean_convdw, 96*4)

s4c2_b2_bn_val_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_val_convdw = readbinfile("./data/stage4_2_branch2_4_running_var.bin", 96)
S4c2_b2_bn_val_convdw = S4c2_b2_bn_val_convdw.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_val_convdw, S4c2_b2_bn_val_convdw, 96*4)

s4c2_b2_bn_gamma_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_gamma_convdw = readbinfile("./data/stage4_2_branch2_4_weight.bin", 96)
S4c2_b2_bn_gamma_convdw = S4c2_b2_bn_gamma_convdw.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_gamma_convdw, S4c2_b2_bn_gamma_convdw, 96*4)

s4c2_b2_bn_beta_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_beta_convdw = readbinfile("./data/stage4_2_branch2_4_bias.bin", 96)
S4c2_b2_bn_beta_convdw = S4c2_b2_bn_beta_convdw.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_beta_convdw, S4c2_b2_bn_beta_convdw, 96*4)

#stage4基本单元2-分支2-1x1普通卷积输出2
out_s4c2_b2_conv2 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4基本单元2-分支2-1x1普通卷积核2，BN参数
s4c2_b2_w_conv2 = xlnk.cma_array(shape = (96, 96, 1, 1), cacheable = 0, dtype = np.float32)
S4c2_b2_w_conv2 = readbinfile("./data/stage4_2_branch2_5_weight.bin", 96*96*1*1)
S4c2_b2_w_conv2 = S4c2_b2_w_conv2.reshape((96, 96, 1, 1))
xlnk.cma_memcopy(s4c2_b2_w_conv2, S4c2_b2_w_conv2, 96*96*1*1*4)

s4c2_b2_bn_mean_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_mean_conv2 = readbinfile("./data/stage4_2_branch2_6_running_mean.bin", 96)
S4c2_b2_bn_mean_conv2 = S4c2_b2_bn_mean_conv2.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_mean_conv2, S4c2_b2_bn_mean_conv2, 96*4)

s4c2_b2_bn_val_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_val_conv2 = readbinfile("./data/stage4_2_branch2_6_running_var.bin", 96)
S4c2_b2_bn_val_conv2 = S4c2_b2_bn_val_conv2.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_val_conv2, S4c2_b2_bn_val_conv2, 96*4)

s4c2_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_gamma_conv2 = readbinfile("./data/stage4_2_branch2_6_weight.bin", 96)
S4c2_b2_bn_gamma_conv2 = S4c2_b2_bn_gamma_conv2.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_gamma_conv2, S4c2_b2_bn_gamma_conv2, 96*4)

s4c2_b2_bn_beta_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c2_b2_bn_beta_conv2 = readbinfile("./data/stage4_2_branch2_6_bias.bin", 96)
S4c2_b2_bn_beta_conv2 = S4c2_b2_bn_beta_conv2.reshape((96))
xlnk.cma_memcopy(s4c2_b2_bn_beta_conv2, S4c2_b2_bn_beta_conv2, 96*4)

#stage4基本单元2-通道合并，通道混洗
in_s4c2_shuff = xlnk.cma_array(shape = (4, 4, 192), cacheable = 0, dtype = np.float32)
out_s4c2_shuff = xlnk.cma_array(shape = (4, 4, 192), cacheable = 0, dtype = np.float32)


#基本单元3
#stage4基本单元3-通道拆分
out_s4c3_ch_spilt1 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
out_s4c3_ch_spilt2 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)

#stage4基本单元3-分支1

#stage4基本单元3-分支2-1x1普通卷积1输出
out_s4c3_b2_conv1 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4基本单元3-分支2-1x1普通卷积1核，BN参数
s4c3_b2_w_conv1 = xlnk.cma_array(shape = (96, 96, 1, 1), cacheable = 0, dtype = np.float32)
S4c3_b2_w_conv1 = readbinfile("./data/stage4_3_branch2_0_weight.bin", 96*96*1*1)
S4c3_b2_w_conv1 = S4c3_b2_w_conv1.reshape((96, 96, 1, 1))
xlnk.cma_memcopy(s4c3_b2_w_conv1, S4c3_b2_w_conv1, 96*96*1*1*4)

s4c3_b2_bn_mean_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_mean_conv1 = readbinfile("./data/stage4_3_branch2_1_running_mean.bin", 96)
S4c3_b2_bn_mean_conv1 = S4c3_b2_bn_mean_conv1.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_mean_conv1, S4c3_b2_bn_mean_conv1, 96*4)

s4c3_b2_bn_val_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_val_conv1 = readbinfile("./data/stage4_3_branch2_1_running_var.bin", 96)
S4c3_b2_bn_val_conv1 = S4c3_b2_bn_val_conv1.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_val_conv1, S4c3_b2_bn_val_conv1, 96*4)

s4c3_b2_bn_gamma_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_gamma_conv1 = readbinfile("./data/stage4_3_branch2_1_weight.bin", 96)
S4c3_b2_bn_gamma_conv1 = S4c3_b2_bn_gamma_conv1.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_gamma_conv1, S4c3_b2_bn_gamma_conv1, 96*4)

s4c3_b2_bn_beta_conv1 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_beta_conv1 = readbinfile("./data/stage4_3_branch2_1_bias.bin", 96)
S4c3_b2_bn_beta_conv1 = S4c3_b2_bn_beta_conv1.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_beta_conv1, S4c3_b2_bn_beta_conv1, 96*4)

#stage4基本单元3-分支2-3x3深度卷积输出
out_s4c3_b2_convdw = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4基本单元3-分支2-3x3深度卷积核，BN参数
s4c3_b2_w_convdw = xlnk.cma_array(shape = (96, 3, 3), cacheable = 0, dtype = np.float32)
S4c3_b2_w_convdw = readbinfile("./data/stage4_3_branch2_3_weight.bin", 96*3*3)
S4c3_b2_w_convdw = S4c3_b2_w_convdw.reshape((96, 3, 3))
xlnk.cma_memcopy(s4c3_b2_w_convdw, S4c3_b2_w_convdw, 96*3*3*4)

s4c3_b2_bn_mean_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_mean_convdw = readbinfile("./data/stage4_3_branch2_4_running_mean.bin", 96)
S4c3_b2_bn_mean_convdw = S4c3_b2_bn_mean_convdw.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_mean_convdw, S4c3_b2_bn_mean_convdw, 96*4)

s4c3_b2_bn_val_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_val_convdw = readbinfile("./data/stage4_3_branch2_4_running_var.bin", 96)
S4c3_b2_bn_val_convdw = S4c3_b2_bn_val_convdw.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_val_convdw, S4c3_b2_bn_val_convdw, 96*4)

s4c3_b2_bn_gamma_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_gamma_convdw = readbinfile("./data/stage4_3_branch2_4_weight.bin", 96)
S4c3_b2_bn_gamma_convdw = S4c3_b2_bn_gamma_convdw.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_gamma_convdw, S4c3_b2_bn_gamma_convdw, 96*4)

s4c3_b2_bn_beta_convdw = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_beta_convdw = readbinfile("./data/stage4_3_branch2_4_bias.bin", 96)
S4c3_b2_bn_beta_convdw = S4c3_b2_bn_beta_convdw.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_beta_convdw, S4c3_b2_bn_beta_convdw, 96*4)

#stage4基本单元3-分支2-1x1普通卷积输出2
out_s4c3_b2_conv2 = xlnk.cma_array(shape = (4, 4, 96), cacheable = 0, dtype = np.float32)
#stage4基本单元3-分支2-1x1普通卷积核2，BN参数
s4c3_b2_w_conv2 = xlnk.cma_array(shape = (96, 96, 1, 1), cacheable = 0, dtype = np.float32)
S4c3_b2_w_conv2 = readbinfile("./data/stage4_3_branch2_5_weight.bin", 96*96*1*1)
S4c3_b2_w_conv2 = S4c3_b2_w_conv2.reshape((96, 96, 1, 1))
xlnk.cma_memcopy(s4c3_b2_w_conv2, S4c3_b2_w_conv2, 96*96*1*1*4)

s4c3_b2_bn_mean_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_mean_conv2 = readbinfile("./data/stage4_3_branch2_6_running_mean.bin", 96)
S4c3_b2_bn_mean_conv2 = S4c3_b2_bn_mean_conv2.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_mean_conv2, S4c3_b2_bn_mean_conv2, 96*4)

s4c3_b2_bn_val_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_val_conv2 = readbinfile("./data/stage4_3_branch2_6_running_var.bin", 96)
S4c3_b2_bn_val_conv2 = S4c3_b2_bn_val_conv2.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_val_conv2, S4c3_b2_bn_val_conv2, 96*4)

s4c3_b2_bn_gamma_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_gamma_conv2 = readbinfile("./data/stage4_3_branch2_6_weight.bin", 96)
S4c3_b2_bn_gamma_conv2 = S4c3_b2_bn_gamma_conv2.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_gamma_conv2, S4c3_b2_bn_gamma_conv2, 96*4)

s4c3_b2_bn_beta_conv2 = xlnk.cma_array(shape = (96), cacheable = 0, dtype = np.float32)
S4c3_b2_bn_beta_conv2 = readbinfile("./data/stage4_3_branch2_6_bias.bin", 96)
S4c3_b2_bn_beta_conv2 = S4c3_b2_bn_beta_conv2.reshape((96))
xlnk.cma_memcopy(s4c3_b2_bn_beta_conv2, S4c3_b2_bn_beta_conv2, 96*4)

#stage4基本单元3-通道合并，通道混洗
in_s4c3_shuff = xlnk.cma_array(shape = (4, 4, 192), cacheable = 0, dtype = np.float32)
out_s4c3_shuff = xlnk.cma_array(shape = (4, 4, 192), cacheable = 0, dtype = np.float32)

print("successfully!")


successfully!


 ## <span style="font-size: 24px;">***conv 5 代码块（核心部分）***</span>

In [15]:
#Conv5卷积核，BN参数
w_conv5 = xlnk.cma_array(shape = (1024, 192, 1, 1), cacheable = 0, dtype = np.float32)
W_conv5 = readbinfile("./data/conv5_0_weight.bin", 1024*192*1*1)
W_conv5 = W_conv5.reshape((1024, 192, 1, 1))
xlnk.cma_memcopy(w_conv5, W_conv5, 1024*192*1*1*4)

bn_mean_conv5 = xlnk.cma_array(shape = (1024), cacheable = 0, dtype = np.float32)
BN_mean_conv5 = readbinfile("./data/conv5_1_running_mean.bin", 1024)
BN_mean_conv5 = BN_mean_conv5.reshape((1024))
xlnk.cma_memcopy(bn_mean_conv5, BN_mean_conv5, 1024*4)

bn_val_conv5 = xlnk.cma_array(shape = (1024), cacheable = 0, dtype = np.float32)
BN_val_conv5 = readbinfile("./data/conv5_1_running_var.bin", 1024)
BN_val_conv5 = BN_val_conv5.reshape((1024))
xlnk.cma_memcopy(bn_val_conv5, BN_val_conv5, 1024*4)

bn_gamma_conv5 = xlnk.cma_array(shape = (1024), cacheable = 0, dtype = np.float32)
BN_gamma_conv5 = readbinfile("./data/conv5_1_weight.bin", 1024)
BN_gamma_conv5 = BN_gamma_conv5.reshape((1024))
xlnk.cma_memcopy(bn_gamma_conv5, BN_gamma_conv5, 1024*4)

bn_beta_conv5 = xlnk.cma_array(shape = (1024), cacheable = 0, dtype = np.float32)
BN_beta_conv5 = readbinfile("./data/conv5_1_bias.bin", 1024)
BN_beta_conv5 = BN_beta_conv5.reshape((1024))
xlnk.cma_memcopy(bn_beta_conv5, BN_beta_conv5, 1024*4)

#Conv5输出4x4x1024
out_conv5 = xlnk.cma_array(shape = (4, 4, 1024), cacheable = 0, dtype = np.float32)
#Globalpool输出1024
out_globalpool = xlnk.cma_array(shape = (1024), cacheable = 0, dtype = np.float32)
#全连接层输出5(对应多少类)
out_fc = xlnk.cma_array(shape = (3), cacheable = 0, dtype = np.float32)
#全连接层权重参数
w_fc = xlnk.cma_array(shape = (3, 1024), cacheable = 0, dtype = np.float32)     #???????????可能是（5，1024）
W_fc = readbinfile("./data/fc_weight.bin", 3*1024)
W_fc = W_fc.reshape((3, 1024))
xlnk.cma_memcopy(w_fc, W_fc, 3*1024*4)

b_fc = xlnk.cma_array(shape = (3), cacheable = 0, dtype = np.float32)
B_fc = readbinfile("./data/fc_bias.bin", 3)
B_fc = B_fc.reshape((3))
xlnk.cma_memcopy(b_fc, B_fc, 3*4)

print("successfully!")

successfully!


 ## <span style="font-size: 24px;">***执行阶段***</span>

In [16]:
def predict(image):
    #Conv1
    hwConv(conv_ip, 0, 3, 2, 1, image, w_conv1, bn_mean_conv1, bn_val_conv1, bn_gamma_conv1, bn_beta_conv1, out_conv1)
    #maxpool
    hwPool(pool_ip, 3, 2, 1, 0, out_conv1, out_maxpool)
    '''                                             stage2                                              '''
    #stage2下采样单元-分支1
    hwConv(conv_ip, 1, 3, 2, 1, out_maxpool, s2s_b1_w_convdw, s2s_b1_bn_mean_convdw,
                    s2s_b1_bn_val_convdw, s2s_b1_bn_gamma_convdw, s2s_b1_bn_beta_convdw, out_s2s_b1_convdw)
    hwConv(conv_ip, 0, 1, 1, 0, out_s2s_b1_convdw, s2s_b1_w_conv1, s2s_b1_bn_mean_conv1,
                    s2s_b1_bn_val_conv1, s2s_b1_bn_gamma_conv1, s2s_b1_bn_beta_conv1, out_s2s_b1_conv1)
    #stage2下采样单元-分支2
    hwConv(conv_ip, 0, 1, 1, 0, out_maxpool, s2s_b2_w_conv1, s2s_b2_bn_mean_conv1,
                    s2s_b2_bn_val_conv1, s2s_b2_bn_gamma_conv1, s2s_b2_bn_beta_conv1, out_s2s_b2_conv1)

    hwConv(conv_ip, 1, 3, 2, 1, out_s2s_b2_conv1, s2s_b2_w_convdw, s2s_b2_bn_mean_convdw,
                    s2s_b2_bn_val_convdw, s2s_b2_bn_gamma_convdw, s2s_b2_bn_beta_convdw, out_s2s_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s2s_b2_convdw, s2s_b2_w_conv2, s2s_b2_bn_mean_conv2,
                    s2s_b2_bn_val_conv2, s2s_b2_bn_gamma_conv2, s2s_b2_bn_beta_conv2, out_s2s_b2_conv2)
    #stage2下采样单元-通道混洗
    #打印分支2第一个BN层（conv1后）的关键参数数值
    in_s2s_shuff[:, :, :24] = out_s2s_b1_conv1
    in_s2s_shuff[:, :, 24:] = out_s2s_b2_conv2
    # 通道混洗硬件加速
    hwch_shuffle(channel_shuffle_ip, 2, in_s2s_shuff, out_s2s_shuff)
    #stage2基本单元1-分支1
    #stage2基本单元1-分支2
    out_s2c1_ch_spilt1 = xlnk.cma_array(shape=(16, 16, 24), cacheable=0, dtype=np.float32)
    out_s2c1_ch_spilt2 = xlnk.cma_array(shape=(16, 16, 24), cacheable=0, dtype=np.float32)
    out_s2c1_ch_spilt1[:, :, :] = out_s2s_shuff[:, :, :24]  # 前24通道
    out_s2c1_ch_spilt2[:, :, :] = out_s2s_shuff[:, :, 24:]  # 后24通道

    hwConv(conv_ip, 0, 1, 1, 0, out_s2c1_ch_spilt2, s2c1_b2_w_conv1, s2c1_b2_bn_mean_conv1,
                    s2c1_b2_bn_val_conv1, s2c1_b2_bn_gamma_conv1, s2c1_b2_bn_beta_conv1, out_s2c1_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s2c1_b2_conv1, s2c1_b2_w_convdw, s2c1_b2_bn_mean_convdw,
                    s2c1_b2_bn_val_convdw, s2c1_b2_bn_gamma_convdw, s2c1_b2_bn_beta_convdw, out_s2c1_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s2c1_b2_convdw, s2c1_b2_w_conv2, s2c1_b2_bn_mean_conv2,
                    s2c1_b2_bn_val_conv2, s2c1_b2_bn_gamma_conv2, s2c1_b2_bn_beta_conv2, out_s2c1_b2_conv2)
    #stage2基本单元1-通道混洗
    in_s2c1_shuff[:, :, :24] = out_s2c1_ch_spilt1
    in_s2c1_shuff[:, :, 24:] = out_s2c1_b2_conv2
    # 通道混洗硬件加速
    hwch_shuffle(channel_shuffle_ip, 2, in_s2c1_shuff, out_s2c1_shuff)

    #stage2基本单元2-分支1
    #stage2基本单元2-分支2
    out_s2c2_ch_spilt1 = xlnk.cma_array(shape=(16, 16, 24), cacheable=0, dtype=np.float32)
    out_s2c2_ch_spilt2 = xlnk.cma_array(shape=(16, 16, 24), cacheable=0, dtype=np.float32)
    out_s2c2_ch_spilt1[:, :, :] = out_s2c1_shuff[:, :, :24]  # 前24通道
    out_s2c2_ch_spilt2[:, :, :] = out_s2c1_shuff[:, :, 24:]  # 后24通道

    hwConv(conv_ip, 0, 1, 1, 0, out_s2c2_ch_spilt2, s2c2_b2_w_conv1, s2c2_b2_bn_mean_conv1,
                    s2c2_b2_bn_val_conv1, s2c2_b2_bn_gamma_conv1, s2c2_b2_bn_beta_conv1, out_s2c2_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s2c2_b2_conv1, s2c2_b2_w_convdw, s2c2_b2_bn_mean_convdw,
                    s2c2_b2_bn_val_convdw, s2c2_b2_bn_gamma_convdw, s2c2_b2_bn_beta_convdw, out_s2c2_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s2c2_b2_convdw, s2c2_b2_w_conv2, s2c2_b2_bn_mean_conv2,
                    s2c2_b2_bn_val_conv2, s2c2_b2_bn_gamma_conv2, s2c2_b2_bn_beta_conv2, out_s2c2_b2_conv2)
    #stage2基本单元2-通道混洗
    in_s2c2_shuff[:, :, :24] = out_s2c2_ch_spilt1
    in_s2c2_shuff[:, :, 24:] = out_s2c2_b2_conv2
    # 通道混洗硬件加速
    hwch_shuffle(channel_shuffle_ip, 2, in_s2c2_shuff, out_s2c2_shuff)
    #stage2基本单元3-分支1
    #stage2基本单元3-分支2
    out_s2c3_ch_spilt1 = xlnk.cma_array(shape=(16, 16, 24), cacheable=0, dtype=np.float32)
    out_s2c3_ch_spilt2 = xlnk.cma_array(shape=(16, 16, 24), cacheable=0, dtype=np.float32)
    out_s2c3_ch_spilt1[:, :, :] = out_s2c2_shuff[:, :, :24]  # 前24通道
    out_s2c3_ch_spilt2[:, :, :] = out_s2c2_shuff[:, :, 24:]  # 后24通道

    hwConv(conv_ip, 0, 1, 1, 0, out_s2c3_ch_spilt2, s2c3_b2_w_conv1, s2c3_b2_bn_mean_conv1,
                    s2c3_b2_bn_val_conv1, s2c3_b2_bn_gamma_conv1, s2c3_b2_bn_beta_conv1, out_s2c3_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s2c3_b2_conv1, s2c3_b2_w_convdw, s2c3_b2_bn_mean_convdw,
                    s2c3_b2_bn_val_convdw, s2c3_b2_bn_gamma_convdw, s2c3_b2_bn_beta_convdw, out_s2c3_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s2c3_b2_convdw, s2c3_b2_w_conv2, s2c3_b2_bn_mean_conv2,
                    s2c3_b2_bn_val_conv2, s2c3_b2_bn_gamma_conv2, s2c3_b2_bn_beta_conv2, out_s2c3_b2_conv2)
    #stage2基本单元3-通道混洗
    in_s2c3_shuff[:, :, :24] = out_s2c3_ch_spilt1
    in_s2c3_shuff[:, :, 24:] = out_s2c3_b2_conv2
    # 通道混洗硬件加速
    hwch_shuffle(channel_shuffle_ip, 2, in_s2c3_shuff, out_s2c3_shuff)
    '''                                             stage3                                              '''
    #stage3下采样单元-分支1
    hwConv(conv_ip, 1, 3, 2, 1, out_s2c3_shuff, s3s_b1_w_convdw, s3s_b1_bn_mean_convdw,
                    s3s_b1_bn_val_convdw, s3s_b1_bn_gamma_convdw, s3s_b1_bn_beta_convdw, out_s3s_b1_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s3s_b1_convdw, s3s_b1_w_conv1, s3s_b1_bn_mean_conv1,
                    s3s_b1_bn_val_conv1, s3s_b1_bn_gamma_conv1, s3s_b1_bn_beta_conv1, out_s3s_b1_conv1)
    #stage3下采样单元-分支2
    hwConv(conv_ip, 0, 1, 1, 0, out_s2c3_shuff, s3s_b2_w_conv1, s3s_b2_bn_mean_conv1,
                    s3s_b2_bn_val_conv1, s3s_b2_bn_gamma_conv1, s3s_b2_bn_beta_conv1, out_s3s_b2_conv1)

    hwConv(conv_ip, 1, 3, 2, 1, out_s3s_b2_conv1, s3s_b2_w_convdw, s3s_b2_bn_mean_convdw,
                    s3s_b2_bn_val_convdw, s3s_b2_bn_gamma_convdw, s3s_b2_bn_beta_convdw, out_s3s_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s3s_b2_convdw, s3s_b2_w_conv2, s3s_b2_bn_mean_conv2,
                    s3s_b2_bn_val_conv2, s3s_b2_bn_gamma_conv2, s3s_b2_bn_beta_conv2, out_s3s_b2_conv2)
    #stage3下采样单元-通道混洗
    in_s3s_shuff[:, :, :48] = out_s3s_b1_conv1
    in_s3s_shuff[:, :, 48:] = out_s3s_b2_conv2
    # 通道混洗硬件加速
    hwch_shuffle(channel_shuffle_ip, 2, in_s3s_shuff, out_s3s_shuff)
    #stage3基本单元1-分支1
    #stage3基本单元1-分支2
    out_s3c1_ch_spilt1[:, :, :] = out_s3s_shuff[:, :, :48]  # 前48通道
    out_s3c1_ch_spilt2[:, :, :] = out_s3s_shuff[:, :, 48:]  # 后48通道
    hwConv(conv_ip, 0, 1, 1, 0, out_s3c1_ch_spilt2, s3c1_b2_w_conv1, s3c1_b2_bn_mean_conv1,
                    s3c1_b2_bn_val_conv1, s3c1_b2_bn_gamma_conv1, s3c1_b2_bn_beta_conv1, out_s3c1_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s3c1_b2_conv1, s3c1_b2_w_convdw, s3c1_b2_bn_mean_convdw,
                    s3c1_b2_bn_val_convdw, s3c1_b2_bn_gamma_convdw, s3c1_b2_bn_beta_convdw, out_s3c1_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s3c1_b2_convdw, s3c1_b2_w_conv2, s3c1_b2_bn_mean_conv2,
                    s3c1_b2_bn_val_conv2, s3c1_b2_bn_gamma_conv2, s3c1_b2_bn_beta_conv2, out_s3c1_b2_conv2)
    #stage3基本单元1-通道混洗
    in_s3c1_shuff[:, :, :48] = out_s3c1_ch_spilt1
    in_s3c1_shuff[:, :, 48:] = out_s3c1_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s3c1_shuff, out_s3c1_shuff)
    #stage3基本单元2-分支1
    #stage3基本单元2-分支2
    out_s3c2_ch_spilt1[:, :, :] = out_s3c1_shuff[:, :, :48]  # 前48通道
    out_s3c2_ch_spilt2[:, :, :] = out_s3c1_shuff[:, :, 48:]  # 后48通道
    hwConv(conv_ip, 0, 1, 1, 0, out_s3c2_ch_spilt2, s3c2_b2_w_conv1, s3c2_b2_bn_mean_conv1,
                    s3c2_b2_bn_val_conv1, s3c2_b2_bn_gamma_conv1, s3c2_b2_bn_beta_conv1, out_s3c2_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s3c2_b2_conv1, s3c2_b2_w_convdw, s3c2_b2_bn_mean_convdw,
                    s3c2_b2_bn_val_convdw, s3c2_b2_bn_gamma_convdw, s3c2_b2_bn_beta_convdw, out_s3c2_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s3c2_b2_convdw, s3c2_b2_w_conv2, s3c2_b2_bn_mean_conv2,
                    s3c2_b2_bn_val_conv2, s3c2_b2_bn_gamma_conv2, s3c2_b2_bn_beta_conv2, out_s3c2_b2_conv2)
    #stage3基本单元2-通道混洗
    in_s3c2_shuff[:, :, :48] = out_s3c2_ch_spilt1
    in_s3c2_shuff[:, :, 48:] = out_s3c2_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s3c2_shuff, out_s3c2_shuff)
    #stage3基本单元3-分支1
    #stage3基本单元3-分支2
    out_s3c3_ch_spilt1[:, :, :] = out_s3c2_shuff[:, :, :48]  # 前48通道
    out_s3c3_ch_spilt2[:, :, :] = out_s3c2_shuff[:, :, 48:]  # 后48通道 
    hwConv(conv_ip, 0, 1, 1, 0, out_s3c3_ch_spilt2, s3c3_b2_w_conv1, s3c3_b2_bn_mean_conv1,
                    s3c3_b2_bn_val_conv1, s3c3_b2_bn_gamma_conv1, s3c3_b2_bn_beta_conv1, out_s3c3_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s3c3_b2_conv1, s3c3_b2_w_convdw, s3c3_b2_bn_mean_convdw,
                    s3c3_b2_bn_val_convdw, s3c3_b2_bn_gamma_convdw, s3c3_b2_bn_beta_convdw, out_s3c3_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s3c3_b2_convdw, s3c3_b2_w_conv2, s3c3_b2_bn_mean_conv2,
                    s3c3_b2_bn_val_conv2, s3c3_b2_bn_gamma_conv2, s3c3_b2_bn_beta_conv2, out_s3c3_b2_conv2)
    #stage3基本单元3-通道混洗
    in_s3c3_shuff[:, :, :48] = out_s3c3_ch_spilt1
    in_s3c3_shuff[:, :, 48:] = out_s3c3_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s3c3_shuff, out_s3c3_shuff)
    #a2=xlnk.cma_stats()
    #print(a2)
    #stage3基本单元4-分支1
    #stage3基本单元4-分支2
    out_s3c4_ch_spilt1[:, :, :] = out_s3c3_shuff[:, :, :48]  # 前48通道
    out_s3c4_ch_spilt2[:, :, :] = out_s3c3_shuff[:, :, 48:]  # 后48通道 
    hwConv(conv_ip, 0, 1, 1, 0, out_s3c4_ch_spilt2, s3c4_b2_w_conv1, s3c4_b2_bn_mean_conv1,
                    s3c4_b2_bn_val_conv1, s3c4_b2_bn_gamma_conv1, s3c4_b2_bn_beta_conv1, out_s3c4_b2_conv1)
    hwConv(conv_ip, 1, 3, 1, 1, out_s3c4_b2_conv1, s3c4_b2_w_convdw, s3c4_b2_bn_mean_convdw,
                    s3c4_b2_bn_val_convdw, s3c4_b2_bn_gamma_convdw, s3c4_b2_bn_beta_convdw, out_s3c4_b2_convdw)
    hwConv(conv_ip, 0, 1, 1, 0, out_s3c4_b2_convdw, s3c4_b2_w_conv2, s3c4_b2_bn_mean_conv2,
                    s3c4_b2_bn_val_conv2, s3c4_b2_bn_gamma_conv2, s3c4_b2_bn_beta_conv2, out_s3c4_b2_conv2)
    #stage3基本单元4-通道混洗
    in_s3c4_shuff[:, :, :48] = out_s3c4_ch_spilt1
    in_s3c4_shuff[:, :, 48:] = out_s3c4_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s3c4_shuff, out_s3c4_shuff)

    #stage3基本单元5-分支1
    #stage3基本单元5-分支2
    out_s3c5_ch_spilt1[:, :, :] = out_s3c4_shuff[:, :, :48]
    out_s3c5_ch_spilt2[:, :, :] = out_s3c4_shuff[:, :, 48:] 
    hwConv(conv_ip, 0, 1, 1, 0, out_s3c5_ch_spilt2, s3c5_b2_w_conv1, s3c5_b2_bn_mean_conv1,
                    s3c5_b2_bn_val_conv1, s3c5_b2_bn_gamma_conv1, s3c5_b2_bn_beta_conv1, out_s3c5_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s3c5_b2_conv1, s3c5_b2_w_convdw, s3c5_b2_bn_mean_convdw,
                    s3c5_b2_bn_val_convdw, s3c5_b2_bn_gamma_convdw, s3c5_b2_bn_beta_convdw, out_s3c5_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s3c5_b2_convdw, s3c5_b2_w_conv2, s3c5_b2_bn_mean_conv2,
                    s3c5_b2_bn_val_conv2, s3c5_b2_bn_gamma_conv2, s3c5_b2_bn_beta_conv2, out_s3c5_b2_conv2)
    #stage3基本单元5-通道混洗
    in_s3c5_shuff[:, :, :48] = out_s3c5_ch_spilt1
    in_s3c5_shuff[:, :, 48:] = out_s3c5_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s3c5_shuff, out_s3c5_shuff)

    #stage3基本单元6-分支1
    #stage3基本单元6-分支2
    out_s3c6_ch_spilt1[:, :, :] = out_s3c5_shuff[:, :, :48]
    out_s3c6_ch_spilt2[:, :, :] = out_s3c5_shuff[:, :, 48:] 
    hwConv(conv_ip, 0, 1, 1, 0, out_s3c6_ch_spilt2, s3c6_b2_w_conv1, s3c6_b2_bn_mean_conv1,
                    s3c6_b2_bn_val_conv1, s3c6_b2_bn_gamma_conv1, s3c6_b2_bn_beta_conv1, out_s3c6_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s3c6_b2_conv1, s3c6_b2_w_convdw, s3c6_b2_bn_mean_convdw,
                    s3c6_b2_bn_val_convdw, s3c6_b2_bn_gamma_convdw, s3c6_b2_bn_beta_convdw, out_s3c6_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s3c6_b2_convdw, s3c6_b2_w_conv2, s3c6_b2_bn_mean_conv2,
                    s3c6_b2_bn_val_conv2, s3c6_b2_bn_gamma_conv2, s3c6_b2_bn_beta_conv2, out_s3c6_b2_conv2)
    #stage3基本单元6-通道混洗
    in_s3c6_shuff[:, :, :48] = out_s3c6_ch_spilt1
    in_s3c6_shuff[:, :, 48:] = out_s3c6_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s3c6_shuff, out_s3c6_shuff)

    #stage3基本单元7-分支1
    #stage3基本单元7-分支2
    out_s3c7_ch_spilt1[:, :, :] = out_s3c6_shuff[:, :, :48]
    out_s3c7_ch_spilt2[:, :, :] = out_s3c6_shuff[:, :, 48:]
    hwConv(conv_ip, 0, 1, 1, 0, out_s3c7_ch_spilt2, s3c7_b2_w_conv1, s3c7_b2_bn_mean_conv1,
                    s3c7_b2_bn_val_conv1, s3c7_b2_bn_gamma_conv1, s3c7_b2_bn_beta_conv1, out_s3c7_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s3c7_b2_conv1, s3c7_b2_w_convdw, s3c7_b2_bn_mean_convdw,
                    s3c7_b2_bn_val_convdw, s3c7_b2_bn_gamma_convdw, s3c7_b2_bn_beta_convdw, out_s3c7_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s3c7_b2_convdw, s3c7_b2_w_conv2, s3c7_b2_bn_mean_conv2,
                    s3c7_b2_bn_val_conv2, s3c7_b2_bn_gamma_conv2, s3c7_b2_bn_beta_conv2, out_s3c7_b2_conv2)
    #stage3基本单元7-通道混洗
    in_s3c7_shuff[:, :, :48] = out_s3c7_ch_spilt1
    in_s3c7_shuff[:, :, 48:] = out_s3c7_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s3c7_shuff, out_s3c7_shuff)
    '''                                             stage4                                              '''
    #stage4下采样单元-分支1
    hwConv(conv_ip, 1, 3, 2, 1, out_s3c7_shuff, s4s_b1_w_convdw, s4s_b1_bn_mean_convdw,
                    s4s_b1_bn_val_convdw, s4s_b1_bn_gamma_convdw, s4s_b1_bn_beta_convdw, out_s4s_b1_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s4s_b1_convdw, s4s_b1_w_conv1, s4s_b1_bn_mean_conv1,
                    s4s_b1_bn_val_conv1, s4s_b1_bn_gamma_conv1, s4s_b1_bn_beta_conv1, out_s4s_b1_conv1)
    #stage4下采样单元-分支2
    hwConv(conv_ip, 0, 1, 1, 0, out_s3c7_shuff, s4s_b2_w_conv1, s4s_b2_bn_mean_conv1,
                    s4s_b2_bn_val_conv1, s4s_b2_bn_gamma_conv1, s4s_b2_bn_beta_conv1, out_s4s_b2_conv1)

    hwConv(conv_ip, 1, 3, 2, 1, out_s4s_b2_conv1, s4s_b2_w_convdw, s4s_b2_bn_mean_convdw,
                    s4s_b2_bn_val_convdw, s4s_b2_bn_gamma_convdw, s4s_b2_bn_beta_convdw, out_s4s_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s4s_b2_convdw, s4s_b2_w_conv2, s4s_b2_bn_mean_conv2,
                    s4s_b2_bn_val_conv2, s4s_b2_bn_gamma_conv2, s4s_b2_bn_beta_conv2, out_s4s_b2_conv2)
    #stage4下采样单元-通道混洗
    in_s4s_shuff[:, :, :96] = out_s4s_b1_conv1
    in_s4s_shuff[:, :, 96:] = out_s4s_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s4s_shuff, out_s4s_shuff)

    #stage4基本单元1-分支1
    #stage4基本单元1-分支2
    out_s4c1_ch_spilt1 [:, :, :]= out_s4s_shuff[:, :, :96]
    out_s4c1_ch_spilt2 [:, :, :]= out_s4s_shuff[:, :, 96:] 
    hwConv(conv_ip, 0, 1, 1, 0, out_s4c1_ch_spilt2, s4c1_b2_w_conv1, s4c1_b2_bn_mean_conv1,
                    s4c1_b2_bn_val_conv1, s4c1_b2_bn_gamma_conv1, s4c1_b2_bn_beta_conv1, out_s4c1_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s4c1_b2_conv1, s4c1_b2_w_convdw, s4c1_b2_bn_mean_convdw,
                    s4c1_b2_bn_val_convdw, s4c1_b2_bn_gamma_convdw, s4c1_b2_bn_beta_convdw, out_s4c1_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s4c1_b2_convdw, s4c1_b2_w_conv2, s4c1_b2_bn_mean_conv2,
                    s4c1_b2_bn_val_conv2, s4c1_b2_bn_gamma_conv2, s4c1_b2_bn_beta_conv2, out_s4c1_b2_conv2)
    #stage4基本单元1-通道混洗
    in_s4c1_shuff[:, :, :96] = out_s4c1_ch_spilt1
    in_s4c1_shuff[:, :, 96:] = out_s4c1_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s4c1_shuff, out_s4c1_shuff)
    #stage4基本单元2-分支1
    #stage4基本单元2-分支2
    out_s4c2_ch_spilt1[:, :, :] = out_s4c1_shuff[:, :, :96]
    out_s4c2_ch_spilt2[:, :, :]= out_s4c1_shuff[:, :, 96:]
    hwConv(conv_ip, 0, 1, 1, 0, out_s4c2_ch_spilt2, s4c2_b2_w_conv1, s4c2_b2_bn_mean_conv1,
                    s4c2_b2_bn_val_conv1, s4c2_b2_bn_gamma_conv1, s4c2_b2_bn_beta_conv1, out_s4c2_b2_conv1)
    hwConv(conv_ip, 1, 3, 1, 1, out_s4c2_b2_conv1, s4c2_b2_w_convdw, s4c2_b2_bn_mean_convdw,
                    s4c2_b2_bn_val_convdw, s4c2_b2_bn_gamma_convdw, s4c2_b2_bn_beta_convdw, out_s4c2_b2_convdw)
    hwConv(conv_ip, 0, 1, 1, 0, out_s4c2_b2_convdw, s4c2_b2_w_conv2, s4c2_b2_bn_mean_conv2,
                    s4c2_b2_bn_val_conv2, s4c2_b2_bn_gamma_conv2, s4c2_b2_bn_beta_conv2, out_s4c2_b2_conv2)
    #stage4基本单元2-通道混洗
    in_s4c2_shuff[:, :, :96] = out_s4c2_ch_spilt1
    in_s4c2_shuff[:, :, 96:] = out_s4c2_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s4c2_shuff, out_s4c2_shuff)
    #stage4基本单元3-分支1
    #stage4基本单元3-分支2
    out_s4c3_ch_spilt1[:, :, :] = out_s4c2_shuff[:, :, :96]
    out_s4c3_ch_spilt2 [:, :, :]= out_s4c2_shuff[:, :, 96:]
    hwConv(conv_ip, 0, 1, 1, 0, out_s4c3_ch_spilt2, s4c3_b2_w_conv1, s4c3_b2_bn_mean_conv1,
                    s4c3_b2_bn_val_conv1, s4c3_b2_bn_gamma_conv1, s4c3_b2_bn_beta_conv1, out_s4c3_b2_conv1)

    hwConv(conv_ip, 1, 3, 1, 1, out_s4c3_b2_conv1, s4c3_b2_w_convdw, s4c3_b2_bn_mean_convdw,
                    s4c3_b2_bn_val_convdw, s4c3_b2_bn_gamma_convdw, s4c3_b2_bn_beta_convdw, out_s4c3_b2_convdw)

    hwConv(conv_ip, 0, 1, 1, 0, out_s4c3_b2_convdw, s4c3_b2_w_conv2, s4c3_b2_bn_mean_conv2,
                    s4c3_b2_bn_val_conv2, s4c3_b2_bn_gamma_conv2, s4c3_b2_bn_beta_conv2, out_s4c3_b2_conv2)
    #stage4基本单元3-通道混洗
    in_s4c3_shuff[:, :, :96] = out_s4c3_ch_spilt1
    in_s4c3_shuff[:, :, 96:] = out_s4c3_b2_conv2
    hwch_shuffle(channel_shuffle_ip, 2, in_s4c3_shuff, out_s4c3_shuff)
    #Conv5
    hwConv(conv_ip, 0, 1, 1, 0, out_s4c3_shuff, w_conv5, bn_mean_conv5, bn_val_conv5, bn_gamma_conv5, bn_beta_conv5, out_conv5)
    #全局池化
    hwPool(pool_ip, 4, 1, 0, 1, out_conv5, out_globalpool)
    #全连接层
    hwfc(fc_ip, out_globalpool, w_fc, b_fc, out_fc)
    #根据out_fc判断结果
    max_val = np.max (out_fc)
    exp_x = np.exp (out_fc - max_val) # 防数值溢出
    sum_exp = np.sum (exp_x)
    out_final = exp_x /sum_exp
    print(out_final)
    # 返回预测类别（0,1,2）
    return np.argmax(out_final)

def process_image(image_path):
    """处理单张图片：读取、标准化"""
    image1 = cv2.imread(image_path).astype(np.float32)
    for r in range(128):
        for c in range(128):
            for ch in range(3):
                if ch == 2:  # B通道
                    image[r][c][0] = ((image1[r][c][ch]/255) - 0.485)/0.229
                if ch == 1:  # G通道
                    image[r][c][1] = ((image1[r][c][ch]/255) - 0.456)/0.224
                if ch == 0:  # R通道
                    image[r][c][2] = ((image1[r][c][ch]/255) - 0.406)/0.225
    return image


def main():
    # 测试集配置：假设三类图片分别放在三个文件夹，命名格式为"class0/xxx.jpg", "class1/xxx.jpg", "class2/xxx.jpg"
    test_sets = []
    classes = ["fire", "ice", "thunder"]  # 类别文件夹名
    for cls_idx, cls_name in enumerate(classes):
        # 获取该类别下所有图片路径
        img_dir = os.path.join("data", cls_name)
        img_paths = [os.path.join(img_dir, f) for f in os.listdir(img_dir) 
                    if f.endswith(('.jpg'))]  # 过滤图片文件
        test_sets.append({"class": cls_idx, "image_paths": img_paths})
    
    total = 0
    correct = 0
    
    for class_info in test_sets:
        true_class = class_info["class"]
        for img_path in class_info["image_paths"]:
            try:
                # 处理图片
                image = process_image(img_path)
                # 预测
                pred_class = predict(image)
                # 统计
                total += 1
                if pred_class == true_class:
                    correct += 1
                print(f"图片: {img_path}  真实类别: {true_class}  预测类别: {pred_class}  {'正确' if pred_class == true_class else '错误'}")
            except Exception as e:
                print(f"处理图片 {img_path} 出错: {e}")
    
    # 计算准确率
    accuracy = correct / total
    print(f"\n测试集总样本数: {total}")
    print(f"正确预测数: {correct}")
    print(f"准确率: {accuracy:.4f}")

if __name__ == "__main__":
    main()

[  9.86484110e-01   6.44213171e-04   1.28717367e-02]
图片: data/fire/fire_1576.jpg  真实类别: 0  预测类别: 0  正确
[  9.92874742e-01   5.76335064e-04   6.54902775e-03]
图片: data/fire/fire_2152.jpg  真实类别: 0  预测类别: 0  正确
[  9.97894704e-01   2.31404740e-09   2.10532360e-03]
图片: data/fire/fire_1659.jpg  真实类别: 0  预测类别: 0  正确
[  9.99502063e-01   1.26962533e-07   4.97825909e-04]
图片: data/fire/fire_1879.jpg  真实类别: 0  预测类别: 0  正确
[  9.99624372e-01   1.87568916e-09   3.75610805e-04]
图片: data/fire/fire_1630.jpg  真实类别: 0  预测类别: 0  正确
[  9.99969602e-01   5.42670699e-14   3.04137848e-05]
图片: data/fire/fire_1889.jpg  真实类别: 0  预测类别: 0  正确
[  9.99544084e-01   2.08015507e-08   4.55893751e-04]
图片: data/fire/fire_0542.jpg  真实类别: 0  预测类别: 0  正确
[  9.88420486e-01   4.37722156e-05   1.15358373e-02]
图片: data/fire/fire_1560.jpg  真实类别: 0  预测类别: 0  正确
[  9.92492020e-01   1.19909728e-05   7.49603892e-03]
图片: data/fire/fire_1967.jpg  真实类别: 0  预测类别: 0  正确
[  9.51823831e-01   4.75733104e-04   4.77003679e-02]
图片: data/fire/fire_1

[  9.89807069e-01   7.52968319e-08   1.01927985e-02]
图片: data/fire/fire_0767.jpg  真实类别: 0  预测类别: 0  正确
[ 0.91977274  0.00155598  0.07867128]
图片: data/fire/fire_1748.jpg  真实类别: 0  预测类别: 0  正确
[  9.99996066e-01   5.14621169e-13   3.98192378e-06]
图片: data/fire/fire_0680.jpg  真实类别: 0  预测类别: 0  正确
[  6.92016423e-01   1.52438781e-06   3.07982057e-01]
图片: data/fire/fire_1662.jpg  真实类别: 0  预测类别: 0  正确
[  9.99998212e-01   2.81814728e-11   1.81336839e-06]
图片: data/fire/fire_1493.jpg  真实类别: 0  预测类别: 0  正确
[  9.99845743e-01   1.15072896e-09   1.54257301e-04]
图片: data/fire/fire_1612.jpg  真实类别: 0  预测类别: 0  正确
[  9.99931455e-01   1.19824836e-11   6.85626437e-05]
图片: data/fire/fire_0134.jpg  真实类别: 0  预测类别: 0  正确
[  7.30190158e-01   1.41133845e-04   2.69668758e-01]
图片: data/fire/fire_0168.jpg  真实类别: 0  预测类别: 0  正确
[  9.76614174e-05   9.99897003e-01   5.41108011e-06]
图片: data/ice/xyxr_images229.jpg  真实类别: 1  预测类别: 1  正确
[  9.75248768e-05   9.99740660e-01   1.61876669e-04]
图片: data/ice/xyxr_images140.jpg

[  4.26516599e-05   9.99900222e-01   5.70579577e-05]
图片: data/ice/xyxr_images1153.jpg  真实类别: 1  预测类别: 1  正确
[ 0.06378776  0.66773003  0.26848218]
图片: data/ice/xyxr_images1223.jpg  真实类别: 1  预测类别: 1  正确
[ 0.00496884  0.99022222  0.00480896]
图片: data/ice/xyxr_images174.jpg  真实类别: 1  预测类别: 1  正确
[ 0.03396314  0.00364943  0.96238744]
图片: data/thunder/374.jpg  真实类别: 2  预测类别: 2  正确
[ 0.0777589   0.02080287  0.90143818]
图片: data/thunder/168 (2).jpg  真实类别: 2  预测类别: 2  正确
[  3.03045614e-04   8.55425696e-06   9.99688387e-01]
图片: data/thunder/228.jpg  真实类别: 2  预测类别: 2  正确
[  1.18243523e-01   5.97806822e-04   8.81158710e-01]
图片: data/thunder/461.jpg  真实类别: 2  预测类别: 2  正确
[ 0.00548757  0.00129464  0.99321777]
图片: data/thunder/68 (2).jpg  真实类别: 2  预测类别: 2  正确
[  7.26389524e-04   1.71663596e-05   9.99256432e-01]
图片: data/thunder/37.jpg  真实类别: 2  预测类别: 2  正确
[  3.07851355e-04   2.20449638e-05   9.99670148e-01]
图片: data/thunder/74 (2).jpg  真实类别: 2  预测类别: 2  正确
[ 0.44057962  0.12191845  0.43750188]
图片: d